In [1]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
from typing import List, Tuple, Callable, Union, Dict, Any, Optional
from collections import defaultdict
from numpy.polynomial.legendre import leggauss
from pyomo.environ import *
from scipy.optimize import linprog
from sympy.core.relational import Relational as SympyRelational
import itertools as itools
from sympy.logic.boolalg import BooleanTrue, BooleanFalse
import numpy as np
import chaospy as cp
import math
import pickle
from pathlib import Path
import sympy as sp
import time
import pandas as pd
import pyomo.environ as pyo
from tqdm.auto import tqdm   # auto-detects notebook vs terminal
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings


c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Smolyak quadrature

In [2]:
def theta_interval_at_point(solution, theta_vector: np.ndarray, max_idx: int = 0, min_idx: int = 1) -> tuple:
    """Given the parametric solution for theta_k and the current 'state' vector (theta_prev + d),
        return the scalar lower and upper bound [t_min, t_max] for this theta_k.

    Args:
        solution (_type_): _description_
        t_vector (np.ndarray): _description_
        max_idx (int, optional): _description_. Defaults to 0.
        min_idx (int, optional): _description_. Defaults to 1.

    Returns:
        tuple: _description_
    """

    theta_vector_aug = np.append(theta_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        theta_min = solution[0]
        theta_max = solution[1]
        return float(theta_min), float(theta_max)

    for region in solution.critical_regions:
        if region.is_inside(theta_vector.reshape(-1, 1)):
            coefficients = np.concatenate([region.A, region.b], axis=1)[:2, :]
            max_coefficients = coefficients[max_idx]
            min_coefficients = coefficients[min_idx]
            # theta_max = float(max_coefficients @ theta_vector_aug)
            # theta_min = float(min_coefficients @ theta_vector_aug)
            theta_max = (max_coefficients @ theta_vector_aug).item()
            theta_min = (min_coefficients @ theta_vector_aug).item()
            return theta_min, theta_max

    raise ValueError(
        "The provided theta_vector is not inside any critical region of the solution.")


def map_u_to_theta_and_jacobian(solutions: List, u: np.ndarray, d_vector: np.ndarray) -> tuple:
    """Given the parametric solutions for theta_k and the current 'state' vector (theta_prev + d),
        return the theta_k vector and the Jacobian matrix dtheta/du.
    Args:
        solutions (List): List of parametric solutions for each theta_k.
        u (np.ndarray): 1D arraay of canonical coordinates 
        d_vector (np.ndarray): Current disturbance vector.
    """
    theta_values = []
    jacobian = 1.0

    for k, sol in enumerate(solutions):
        if isinstance(d_vector, np.ndarray):
            theta_vector = np.block([np.array(theta_values), d_vector])
        else:
            theta_vector = np.array(theta_values, dtype=float)

        theta_min, theta_max = theta_interval_at_point(sol, theta_vector)
        length = theta_max - theta_min

        theta_k = 0.5 * length * u[k] + 0.5 * (theta_max + theta_min)
        theta_values.append(theta_k)

        jacobian *= 0.5 * length

    return np.array(theta_values, dtype=float), jacobian


def calculate_stocflexibility_smolyak(solutions: List, level: int, joint_func: Callable[[List[float]], float], d_vector: np.ndarray = None, rule: str = "gaussian") -> float:
    """Compute stochastic flexibility using a Smolyak sparse grid in canonical  u-space

    Args:
        solutions (List): list of solutions for each theta dimension (same structure as in calculate_stocflexibility)
        level (int): Smolyak level (1,2,3,...) controls accuracy & number of points
        joint_func (Callable[[List[float]], float]): callable f(theta_list) -> scalar
        d_vector (np.ndarray, optional):design vector (np.ndarray). Defaults to None.
        rule (str, optional): 1D quadrature rule passed to chaospy (e.g. "gaussian"). Defaults to "gaussian".

    Returns:
        float: _description_
    """

    n_theta = len(solutions)

    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    nodes_u, weights_expectation = cp.quadrature.sparse_grid(
        order=level, dist=dist, rule=rule)

    weights_u = weights_expectation * (2.0 ** n_theta)

    nodes_u = nodes_u.T

    start = time.time()
    stochastic_flexibility = 0.0

    for i in range(nodes_u.shape[0]):
        u_vector = nodes_u[i, :]
        theta_vector, jacobian = map_u_to_theta_and_jacobian(
            solutions, u_vector, d_vector)
        func_value = joint_func(theta_vector)
        stochastic_flexibility += func_value * jacobian * weights_u[i]

    end = time.time()
    print(
        f"Smolyak stochastic flexibility computed in {end - start:.4f} seconds.")
    return stochastic_flexibility

## Gaussian Legendre quadrature

In [3]:
def gauss_legendre_between_bounds(expr_coeffs: np.ndarray, n_gl: int, max_idx: int = 0, min_idx: int = 1):
    """
    Generate n Gauss–Legendre quadrature points and weights between min and max bounds
    defined by two linear expressions.

    Parameters:
        expr_coeffs (np.ndarray): 2xD array. Row 0 = max point co`efficients, Row 1 = min.
        n (int): Number of quadrature points.

    Returns:
        points (np.ndarray): (n, D) array of quadrature points.
        weights (np.ndarray): (n,) array of weights.
    """
    if expr_coeffs.shape[0] != 2:
        raise ValueError("expr_coeffs must have two rows")

    max_coeffs = expr_coeffs[max_idx]
    min_coeffs = expr_coeffs[min_idx]

    # Get Gauss–Legendre points and weights on [-1, 1]
    nodes, weights = leggauss(n_gl)
    weights = weights.reshape(-1, 1)

    # Affine transformation to domain [min_coeffs, max_coeffs]
    points = 0.5 * (np.outer((nodes + 1), max_coeffs) +
                    np.outer((1 - nodes), min_coeffs))

    # Adjust weights to match new domain
    weights = 0.5 * weights@(max_coeffs - min_coeffs).reshape(1, -1)

    return points, weights


def get_quadrature_points(solution, nq: int, t_vector: np.ndarray):
    # Augment t_vector once
    t_vector_aug = np.append(t_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        qpoints, qweights = np.polynomial.legendre.leggauss(nq)
        min, max = solution[0], solution[1]
        qps_mapped = 0.5*(max*(1+qpoints) + min*(1-qpoints))
        qws_mapped = 0.5*(max-min)*qweights
        # print(max, min, qps_mapped, qws_mapped)
        return max, min, qps_mapped, qws_mapped

    for region in solution.critical_regions:
        if region.is_inside(t_vector.reshape(-1, 1)):
            coeffs = np.concatenate([region.A, region.b], axis=1)[:2, :]
            qpoints, qweights = gauss_legendre_between_bounds(
                expr_coeffs=coeffs, n_gl=nq)
            return coeffs[0] @ t_vector_aug, coeffs[1] @ t_vector_aug, qpoints @ t_vector_aug, qweights @ t_vector_aug

    # print(f't_vector: {t_vector}')
    # print(f'solution:{solution}')
    raise ValueError("No region found that contains the given t_vector.")


def calculate_stocflexibility(
    sols,
    nq: Union[int, list],
    joint_func,
    d_vector: np.ndarray = None,
    verbose: bool = True,
):
    # Validate nq if it's a list
    if isinstance(nq, list):
        if len(nq) != len(sols):
            raise ValueError(
                "If nq is a list, it must have the same length as sols")

    start_time = time.perf_counter()

    def recurse(level: int, theta_prev: list, weight_prev: float) -> float:
        if level == len(sols):
            return weight_prev * joint_func(theta_prev)

        nql = nq[level] if isinstance(nq, list) else nq

        t_vector = (
            np.block([np.array(theta_prev), d_vector])
            if isinstance(d_vector, np.ndarray)
            else np.array(theta_prev)
        )

        _, _, t_points, t_weights = get_quadrature_points(
            solution=sols[level],
            nq=nql,
            t_vector=t_vector,
        )

        t_points = t_points.flatten()
        t_weights = t_weights.flatten()

        return sum(
            recurse(level + 1, theta_prev + [v], weight_prev * w)
            for v, w in zip(t_points, t_weights)
        )

    stflex = recurse(level=0, theta_prev=[], weight_prev=1.0)

    end_time = time.perf_counter()

    if verbose:
        print(f"Gaussian Legendre Stochastic Flexibility Elapsed time: {end_time - start_time:.4f} s"
              )

    return stflex

In [4]:
def mpformulate_theta_bounds(flex_sol, num_theta: int, theta_bounds: list, num_design: int = 0, design_bounds: list = None, psi_idx: int = 0, theta_m: int = 0):
    A0, b0, F0 = np.empty((len(flex_sol), num_theta)), np.empty(
        (len(flex_sol), 1)), np.empty((len(flex_sol), num_design))
    num_cr = len(flex_sol.critical_regions)
    for i, region in enumerate(flex_sol.critical_regions):
        A0[i] = region.A[psi_idx, :num_theta]
        b0[i] = -region.b[psi_idx]
        F0[i] = -region.A[psi_idx, num_theta:num_theta+num_design]
    # print(f'num_cr:{num_cr}')
    # print(f'num_theta:{num_theta}')
    # print(f'num_design:{num_design}')
    # print(f"A0: {A0}")
    # print(f"b0: {b0}")
    # print(f"F0: {F0}")

    c = np.hstack([np.array([-1, 1]).reshape(1, -1),
                  np.zeros((1, 2 * (num_theta - 1 - theta_m)))]).reshape(-1, 1)
    # print(f'c:{c}')
    # print(f'c.shape: {c.shape}')

    row1_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (A0[:, [i]], np.zeros((num_cr, 1)))])
    row2_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (np.zeros((num_cr, 1)), A0[:, [i]])])
    bound_row = np.hstack([np.array([-1, 1]).reshape(1, -1),
                          np.zeros((1, 2 * (num_theta - 1 - theta_m)))])
    A = np.vstack([row1_block, row2_block, bound_row, -
                  np.eye(2*(num_theta-theta_m)), np.eye(2*(num_theta-theta_m))])
    # print(f'A: {A}')
    # print(f'A.shape: {A.shape}')

    x_lb = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][0]] * 2])
    x_ub = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][1]] * 2])
    b = np.vstack([b0, b0, np.zeros((1, 1)), -
                  x_lb.reshape(-1, 1), x_ub.reshape(-1, 1)])
    # print(f'b: {b}')
    # print(f'b.shape: {b.shape}')

    if F0.size == 0 and theta_m == 0:
        # print('here')
        return A, b, c, np.array([]), np.array([]), np.array([]), np.array([])

    F = np.vstack([F0, F0, np.zeros((1, num_design)), np.zeros(
        (4*(num_theta-theta_m), num_design))]) if num_design > 0 else np.vstack([F0, F0])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    if theta_m > 0:
        F_lltheta = np.hstack([A0[:, [i]] for i in range(theta_m)])
        # print(f'F_lltheta: {F_lltheta}')
        # print(f'F_lltheta.shape: {F_lltheta.shape}')
        F = np.hstack([np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))]), F]
                      ) if F.size > 0 else np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')

    H = np.zeros((2*(num_theta-theta_m), theta_m+num_design))
    # print(f'H:{H}')
    # print(f'H.shape: {H.shape}')

    A_t = np.vstack([-np.eye(theta_m+num_design), np.eye(theta_m+num_design)])
    # print(f'A_t:{A_t}')
    # print(f'A_t.shape: {A_t.shape}')

    theta_lb = np.array([-theta_bounds[i][0] for i in range(theta_m)] + ([-j[0] for j in design_bounds] if isinstance(design_bounds, list)
                                                                         else [])).reshape(-1, 1)
    theta_ub = np.array([theta_bounds[i][1] for i in range(theta_m)] + ([j[1] for j in design_bounds] if isinstance(design_bounds, list)
                                                                        else [])).reshape(-1, 1)

    b_t = np.vstack([theta_lb, theta_ub])
    # print(f'b_t:{b_t}')
    # print(f'b_t.shape: {b_t.shape}')

    return A, b, c, H, A_t, b_t, F

In [5]:
# def get_theta_bounds(flex_sol, numt, tbounds, numd: int = 0, dbounds: list = None, mp_algo: mpqp_algorithm = mpqp_algorithm.combinatorial):
#     theta_bound_dict = defaultdict(dict)
#     prob_dict = defaultdict(dict)
#     for i in range(numt):
#         A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(
#             flex_sol=flex_sol, num_theta=numt, num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
#         # print(f'A.shape:{A.shape}')
#         # print(f'b.shape: {b.shape}')
#         # print(f'F.shape: {F.shape}')
#         if F.size != 0:
#             prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)
#             prob.process_constraints()
#             solution = solve_mpqp(problem=prob, algorithm=mp_algo)
#             prob_dict[f't{i}'] = prob
#             theta_bound_dict[f't{i}'] = solution
#         else:
#             linsol = linprog(c=c, A_ub=A, b_ub=b)
#             prob_dict[f't{i}'] = linsol
#             theta_bound_dict[f't{i}'] = [linsol.x[1], linsol.x[0]]
#             # if linsol.success:
#             # print("Optimal value:", linsol.fun)
#             # print("Optimal x:", linsol.x)
#         print(f'Finished solving for theta{i+1}')
#     probs = [p for key, p in prob_dict.items()]
#     sols = [sol for key, sol in theta_bound_dict.items()]
# 
#     return probs, sols

In [6]:
def get_theta_bounds(flex_sol, numt, tbounds, numd: int = 0, dbounds: list = None, mp_algo: mpqp_algorithm = mpqp_algorithm.combinatorial):
    theta_bound_dict = defaultdict(dict)
    prob_dict = defaultdict(dict)

    for i in range(numt):
        A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(
            flex_sol=flex_sol, num_theta=numt, num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
        # print(f'A.shape:{A.shape}')
        # print(f'b.shape: {b.shape}')
        # print(f'F.shape: {F.shape}')
        if F.size != 0:
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("error", category=UserWarning, message="The chebychev ball has either a radius of zero, or the problem is not feasible!")
                    prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)

                prob.process_constraints()
                solution = solve_mpqp(problem=prob, algorithm=mp_algo)

                prob_dict[f"t{i}"] = prob
                theta_bound_dict[f"t{i}"] = solution

            except UserWarning as w:
                # MPLP infeasible / degenerate
                print(f"[theta {i}] MPLP infeasible / zero Chebyshev ball: {w}")
                prob_dict[f"t{i}"] = None
                theta_bound_dict[f"t{i}"] = None

        else:
            # LP fallback branch via scipy.optimize.linprog
            linres = linprog(c=c, A_ub=A, b_ub=b)

            if not linres.success:
                print(f"[theta {i}] linprog failed: status={linres.status}, "f"message={linres.message}")
                prob_dict[f"t{i}"] = linres
                theta_bound_dict[f"t{i}"] = None
            else:
                prob_dict[f"t{i}"] = linres
                theta_bound_dict[f"t{i}"] = [linres.x[1], linres.x[0]]

        print(f"Finished solving for theta{i+1}")

    probs = [p for _, p in prob_dict.items()]
    sols = [sol for _, sol in theta_bound_dict.items()]
    return probs, sols

In [7]:
def create_discrete_state_list(ny:int , states: list = None, unallowed_states: list[tuple] = None) -> list:
    
    states = states or [[0, 1] for _ in range(ny)]
    unallowed_set = set(unallowed_states or ())
    
    if isinstance(states[0], list) and len(states) != ny:
        ValueError(f"{len(states)} != {ny}. Number of lists within states should equal ny")
    
    return [s for s in itools.product(*states) if s not in unallowed_set]

In [8]:
def joint_state_prob_dict(
    dicts: list[dict],
    unallowed_states: list[tuple] | None = None,
    renormalize: bool = False,
    check_probs: bool = True,
    tol: float = 1e-12,
) -> dict[tuple, float]:
    """
    Parameters
    ----------
    dicts : list[dict]
        Each dict maps state_value -> probability for one variable.
    unallowed_states : list[tuple] | None
        State tuples to exclude, e.g. [(0,1,1), (1,0,0)].
    renormalize : bool
        If True, renormalize remaining probabilities to sum to 1
        after removing unallowed states.
    check_probs : bool
        If True, verify each input dictionary sums to 1.
    tol : float
        Tolerance used for probability-sum checks.

    Returns
    -------
    dict[tuple, float]
        Mapping from state tuple -> probability
    """
    if not dicts:
        return {}

    unallowed_set = set(unallowed_states or ())

    if check_probs:
        for i, d in enumerate(dicts):
            s = sum(d.values())
            if not math.isclose(s, 1.0, rel_tol=0.0, abs_tol=tol):
                raise ValueError(f"dicts[{i}] probabilities sum to {s}, not 1")

    vp_lists = [list(d.items()) for d in dicts]

    result = {}
    for combo in itools.product(*vp_lists):
        state = tuple(v for v, _ in combo)

        if state in unallowed_set:
            continue

        prob = 1.0
        for _, p in combo:
            prob *= p

        result[state] = prob

    if renormalize:
        total = sum(result.values())
        if math.isclose(total, 0.0, rel_tol=0.0, abs_tol=tol):
            raise ValueError("No probability mass remains after removing unallowed states")
        result = {state: p / total for state, p in result.items()}

    return result

In [9]:
# # Bansal (2000) Illustrative Example
# t_bounds = [(0, 4), (0, 4)]
# d_bounds = [(0, 5), (0, 5)]
# nt = len(t_bounds)
# nd = len(d_bounds)

# m = MPModeler()

# u = m.add_var(name='u')
# x = m.add_var(name='x')
# z = m.add_var(name='z')

# t1 = m.add_param(name='t1')
# t2 = m.add_param(name='t2')
# d1 = m.add_param(name='d1')
# d2 = m.add_param(name='d2')
# m.add_constr(2*x - 3*z + t1 - d2 == 0)
# m.add_constr(x - z/2 - t1/2 + t2/2 + d1 - 7*d2/2 <= u)
# m.add_constr(-2*x + 2*z - 4*t1/3 - t2 + 2*d2 + 1/3 <= u)
# m.add_constr(-x + 5*z/2 + t1/2 - t2 - d1 + d2/2 - 1 <= u)
# m.add_constr(-50 <= x)
# m.add_constr(-50 <= z)
# m.add_constr(t_bounds[0][0] <= t1)
# m.add_constr(t_bounds[1][0] <= t2)
# m.add_constr(d_bounds[0][0] <= d1)
# m.add_constr(d_bounds[1][0] <= d2)
# m.add_constr(t1 <= t_bounds[0][1])
# m.add_constr(t2 <= t_bounds[1][1])
# m.add_constr(d1 <= d_bounds[0][1])
# m.add_constr(d2 <= d_bounds[1][1])
# m.set_objective(u)
# prob = m.formulate_problem()
# prob.process_constraints()

# solution_flexibility = solve_mpqp(
#     problem=prob, algorithm=mpqp_algorithm.geometric)

# start_time = time.time()
# prob_list, sol_list = get_theta_bounds(
#     flex_sol=solution_flexibility, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds)
# end_time = time.time()
# print(f'Elapsed time for solving mp problems: {end_time-start_time}')


# def joint_pdf(theta: list):
#     return (2/np.pi)*np.exp(-2*((theta[0]-2)**2 + (theta[1]-2)**2))

In [10]:
y_dict = {
    (0,0,0): 0.001,
    (0,0,1): 0.003,
    (0,1,0): 0.006,
    (1,0,0): 0.010,
    (0,1,1): 0.040,
    (1,0,1): 0.066,
    (1,1,0): 0.114,
    (1,1,1): 0.760
}

In [11]:
# y1_val = 1
# y2_val = 1
# y3_val = 1

t_bounds = [(8, 16), (3, 11)]
d_bounds = [(0, 10), (0, 10), (0, 10)]
nt = len(t_bounds)
nd = len(d_bounds)

In [12]:
def create_flexibility_model(y_list: tuple = None):
    j1 = 0.92
    j2 = 0.85
    j3 = 0.75
    
    m = MPModeler()
    
    u = m.add_var(name='u')
    F1 = m.add_var(name="F1")
    F2 = m.add_var(name="F2")
    F3 = m.add_var(name="F3")
    F4 = m.add_var(name="F4")
    F5 = m.add_var(name="F5")
    F6 = m.add_var(name="F6")
    F7 = m.add_var(name="F7")
    
    S = m.add_param(name='S')
    D = m.add_param(name='D')
    
    d1 = m.add_param(name='d1')
    d2 = m.add_param(name='d2')
    d3 = m.add_param(name='d3')
    
    
    m.add_constr(F4 - j1 * F2 == 0)
    m.add_constr(F1 - F2 - F3 == 0)
    m.add_constr(F5 - j2 * F4 == 0)
    m.add_constr(F6 - j3 * F3 == 0)
    m.add_constr(F7 - F5 - F6 == 0)
    m.add_constr(F1 - S <= u)
    m.add_constr(D - F7 <= u)
    m.add_constr(F2 - d1 * y_list[0] <= u)
    m.add_constr(F4 - d2 * y_list[1] <= u)
    m.add_constr(F3 - d3 * y_list[2] <= u)
    
    # for v in [F1, F2, F3, F4, F5, F6, F7]:
    #     m.add_constr(v >= 0)
    
    m.add_constr(t_bounds[0][0] + 1e-6 <= S)
    m.add_constr(S <= t_bounds[0][1])
    m.add_constr(t_bounds[1][0] + 1e-6 <= D)
    m.add_constr(D <= t_bounds[1][1])
    
    m.add_constr(d_bounds[0][0] <= d1)
    m.add_constr(d1 <= d_bounds[0][1])
    m.add_constr(d_bounds[1][0] <= d2)
    m.add_constr(d2 <= d_bounds[1][1])
    m.add_constr(d_bounds[2][0] <= d3)
    m.add_constr(d3 <= d_bounds[2][1])
    
    m.set_objective(u)
    
    return m

In [13]:
y_state_list = list(y_dict.keys())

In [14]:
y_state_list

[(0, 0, 0),
 (0, 0, 1),
 (0, 1, 0),
 (1, 0, 0),
 (0, 1, 1),
 (1, 0, 1),
 (1, 1, 0),
 (1, 1, 1)]

In [15]:
flex_model = create_flexibility_model(y_list=y_state_list[0])

In [16]:
prob = flex_model.formulate_problem()
prob.process_constraints()

Set parameter Username
Academic license - for non-commercial use only - expires 2027-02-27


In [17]:
solution_flexibility = solve_mpqp(problem=prob, algorithm=mpqp_algorithm.geometric_parallel)

Using a found active set [0, 1, 2, 3, 4, 6, 7, 8]
Spawned threads across 24
 Number of Facets to look at this time 10


In [18]:
start_time = time.time()
prob_list, sol_list = get_theta_bounds(flex_sol=solution_flexibility, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds, mp_algo=mpqp_algorithm.geometric_parallel)
end_time = time.time()
print(f'Elapsed time for solving mp problems: {end_time-start_time}')

[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
Elapsed time for solving mp problems: 0.004605531692504883


In [19]:
def joint_pdf(theta: list):
    Sval, Dval = theta
    eps = 1e-12
    # x = max(Sval - 8.0, eps)
    return (1/(1.2 * np.pi * (Sval - 8.0))) * np.exp(
        -1.39*(np.log(Sval - 8.0))**2 - 0.5*(Dval - 7.0)**2)

In [20]:
# # d_vector = np.array([5, 7, 9])
# d_vector = np.array([7, 7, 7])
# 
# # REDUCE nq
# nq = 10
# 
# sf_idx_gaussian = calculate_stocflexibility(sols=sol_list, nq=nq, joint_func=joint_pdf, d_vector=d_vector)
# sf_idx_smolyak = calculate_stocflexibility_smolyak(solutions=sol_list, level=16, joint_func=joint_pdf, d_vector=d_vector)
# 
# print(f'Stochastic Flexibility Index Gaussian Legendre quadrature: {sf_idx_gaussian:.4}')
# print(f'Stochastic Flexibility Index Smolyak quadrature: {sf_idx_smolyak:.4}')

## SF Expression

In [21]:
def get_bounds_regions(sols: List, min_idx: int = 1, max_idx: int = 0):
    theta_bounds_list = []
    theta_regions_list = []

    for theta_sol in sols:
        if theta_sol is None:
            theta_bounds_list.append(None)
            theta_regions_list.append(None)
            continue
        
        if not getattr(theta_sol, "critical_regions", None):
            theta_bounds_list.append(None)
            theta_regions_list.append(None)
            continue
        
        min_max_list = []
        region_list = []

        for cr in theta_sol.critical_regions:
            # Store bounds
            Ab = np.concatenate([cr.A, cr.b], axis=1)[:2]
            min_max_list.append([Ab[min_idx].tolist(), Ab[max_idx].tolist()])

            # Store region constraints
            Ef = np.concatenate([cr.E, -cr.f], axis=1)
            region_array = np.array([row.tolist() for row in Ef], dtype=float)
            region_list.append(region_array)

        # Append per-theta data
        theta_bounds_list.append(np.array(min_max_list))
        # <-- each region is a 2D array
        theta_regions_list.append(np.array(region_list, dtype=object))

    return theta_bounds_list, theta_regions_list


def generate_region_combos(region_sizes, n_gl):
    """Generate region index combinations based on critical region structure."""
    n_theta = len(region_sizes)
    region_combo_shape = []
    for k in range(n_theta):
        n_paths = int(np.prod(n_gl[:k])) if k > 0 else 1
        region_combo_shape.extend([range(region_sizes[k])] * n_paths)
    return list(itools.product(*region_combo_shape))


def affine_expr(coeffs, symbols):
    return sum(c * s for c, s in zip(coeffs[:-1], symbols)) + coeffs[-1]


def normalized_lhs(ineq):
    return ineq.lhs.expand() if hasattr(ineq, 'lhs') else None

In [22]:
def _prepare_state_data(state, *, solve_algo, theta_algo, log=False):
    """
    Build and solve the flexibility problem for one state, then return only valid theta-related data.

    Returns
    -------
    dict with keys:
        state
        flex_sol
        sol_list
        theta_bounds_list
        theta_regions_list
        filtered_solutions
        filtered_theta_bounds
        filtered_theta_regions
    or None if no valid theta regions exist.
    """
    state_model = create_flexibility_model(y_list=state)
    state_prob = state_model.formulate_problem()
    state_prob.process_constraints()

    flex_sol = solve_mpqp(problem=state_prob, algorithm=solve_algo)

    if log:
        print(f"Number of critical regions in for flexibility function for state {state}: {len(flex_sol.critical_regions)}")

    _, sol_list = get_theta_bounds(flex_sol=flex_sol, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds, mp_algo=theta_algo)

    theta_bounds_list, theta_regions_list = get_bounds_regions(sols=sol_list)

    filtered = [(sol, tb, tr) for sol, tb, tr in zip(sol_list, theta_bounds_list, theta_regions_list) if sol is not None and tb is not None and tr is not None]

    if not filtered:
        return None

    filtered_solutions = [sol for sol, _, _ in filtered]
    filtered_theta_bounds = [tb for _, tb, _ in filtered]
    filtered_theta_regions = [tr for _, _, tr in filtered]

    return {
        "state": state,
        "flex_sol": flex_sol,
        "sol_list": sol_list,
        "theta_bounds_list": theta_bounds_list,
        "theta_regions_list": theta_regions_list,
        "filtered_solutions": filtered_solutions,
        "filtered_theta_bounds": filtered_theta_bounds,
        "filtered_theta_regions": filtered_theta_regions,
    }

In [23]:
def _safe_sf_call(func, state, label, **kwargs):
    try:
        return func(**kwargs)
    except ValueError as e:
        if "not inside any critical region" in str(e):
            print(f"Skipping {label} SF for state {state}: {e}")
            return None
        raise

In [24]:
def calculate_esf(y_d: dict, d_v, n_q: int, s_level: int):
    g_esf, s_esf = 0, 0

    for state, prob in y_d.items():
        try:
            data = _prepare_state_data(state, solve_algo=mpqp_algorithm.combinatorial_parallel, theta_algo=mpqp_algorithm.combinatorial_parallel)

            if data is None:
                print(f"No valid theta regions for state {state}; skipping state")
                continue

            sols = data["filtered_solutions"]

            sf_idx_gaussian = _safe_sf_call(calculate_stocflexibility, state=state, label="Gaussian", sols=sols, nq=n_q, joint_func=joint_pdf, d_vector=d_v)
            if sf_idx_gaussian is not None:
                g_esf += sf_idx_gaussian * prob

            sf_idx_smolyak = _safe_sf_call(calculate_stocflexibility_smolyak, state=state, label="Smolyak", solutions=sols, level=s_level, joint_func=joint_pdf,
                                           d_vector=d_v)
            if sf_idx_smolyak is not None:
                s_esf += sf_idx_smolyak * prob

        except ValueError as e:
            print(f"Skipping state {state} due to ValueError: {e}")
            continue

        print(f"Finished for state {state}.")

    return g_esf, s_esf

In [25]:
# def calculate_esf(y_d: dict, d_v, n_q: int, s_level: int):
#     g_esf, s_esf = 0, 0
# 
#     for s in y_d:
#         try:
#             s_model = create_flexibility_model(y_list=s)
#             s_prob = s_model.formulate_problem()
#             s_prob.process_constraints()
# 
#             s_flex_sol = solve_mpqp(problem=s_prob, algorithm=mpqp_algorithm.combinatorial_parallel)
#             # print(f"Number of critical regions in for flexibility function for state {s}: {len(s_flex_sol.critical_regions)}")
# 
#             _, s_sol_list = get_theta_bounds(flex_sol=s_flex_sol, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds, mp_algo=mpqp_algorithm.combinatorial_parallel)
#             s_theta_bounds_list, s_theta_regions_list = get_bounds_regions(sols=s_sol_list)
# 
#             filtered = [(sol, tb, tr) for sol, tb, tr in zip(s_sol_list, s_theta_bounds_list, s_theta_regions_list)
#                 if sol is not None and tb is not None and tr is not None]
# 
#             if not filtered:
#                 print(f"No valid theta regions for state {s}; skipping state")
#                 continue
# 
#             filtered_solutions = [sol for sol, _, _ in filtered]
# 
#             sf_idx_gaussian = _safe_sf_call(calculate_stocflexibility, state=s, label="Gaussian", sols=filtered_solutions, nq=n_q, joint_func=joint_pdf, d_vector=d_v)
#             if sf_idx_gaussian is not None:
#                 g_esf += sf_idx_gaussian * y_d[s]
# 
#             sf_idx_smolyak = _safe_sf_call(calculate_stocflexibility_smolyak, state=s, label="Smolyak", solutions=filtered_solutions, level=s_level,
#                 joint_func=joint_pdf, d_vector=d_v)
#             if sf_idx_smolyak is not None:
#                 s_esf += sf_idx_smolyak * y_d[s]
# 
#         except ValueError as e:
#             print(f"Skipping state {s} due to ValueError: {e}")
#             continue
# 
#         print(f"Finished for state {s}.")
# 
#     return g_esf, s_esf

In [26]:
# design_vector = np.array([7,7,7])
# design_vector = np.array([3.8363, 3.5294, 4.1912])
# design_vector = np.array([5,7,9])
# design_vector = np.array([4,4,5])
design_vector = np.array([3.836317135544959, 3.529411764699034, 4.0])

In [27]:
gl_esf, sm_esf = calculate_esf(y_d=y_dict, d_v=design_vector, n_q = 10, s_level = 8)

Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 4
Time to run all tasks in parallel 0.20553255081176758
Time to process all depth outputs 0.0
Time at depth test 2, 0.20653271675109863
Number of active sets to be considered is 3
Time to run all tasks in parallel 0.19643878936767578
Time to process all depth outputs 0.0
Time at depth test 3, 0.4029715061187744
Number of active sets to be considered is 2
Time to run all tasks in parallel 0.20577239990234375
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
No valid theta regions for state (0, 0, 0); skipping state
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 5
Time to run all tasks in p

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


Smolyak stochastic flexibility computed in 0.0090 seconds.
Finished for state (0, 0, 1).
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 4
Time to run all tasks in parallel 0.00506281852722168
Time to process all depth outputs 0.0
Time at depth test 2, 0.00506281852722168
Number of active sets to be considered is 3
Time to run all tasks in parallel 0.004006147384643555
Time to process all depth outputs 0.0
Time at depth test 3, 0.009068965911865234
Number of active sets to be considered is 2
Time to run all tasks in parallel 0.008479118347167969
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
No valid theta regions for state (0, 1, 0); skipping state
Spawned threads across 24
Tim

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


Smolyak stochastic flexibility computed in 0.0085 seconds.
Finished for state (0, 1, 1).
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 5
Time to run all tasks in parallel 0.006964921951293945
Time to process all depth outputs 0.0
Time at depth test 2, 0.006964921951293945
Number of active sets to be considered is 6
Time to run all tasks in parallel 0.005996227264404297
Time to process all depth outputs 0.0
Time at depth test 3, 0.012961149215698242
Number of active sets to be considered is 10
Time to run all tasks in parallel 0.01561594009399414
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 15
Time to run all tasks in parallel 0.01600027084350586
Time to process all depth outputs 0.0
Time at depth test 2, 0.01600027084350586
Number of active sets to be considered is 78
Time to run all tasks in parallel 0.07618856430053711
Time to process all depth outputs 0.0
Time at depth test 3, 0.09218883

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


Time to run all tasks in parallel 0.006198406219482422
Time to process all depth outputs 0.0
Time at depth test 2, 0.006198406219482422
Number of active sets to be considered is 6
Time to run all tasks in parallel 0.007619142532348633
Time to process all depth outputs 0.0
Time at depth test 3, 0.013817548751831055
Number of active sets to be considered is 10
Time to run all tasks in parallel 0.015053272247314453
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 15
Time to run all tasks in parallel 0.016001224517822266
Time to process all depth outputs 0.0
Time at depth test 2, 0.016001224517822266
Number of active sets to be considered is 78
Time to run all tasks in parallel 0.07643437385559082
Time to process all depth outputs 0.0
Time at depth test 3, 0.09243559837341309
Number of active sets to be considered is 318
Time to run all tasks in parallel 0.08408546447753906
Time to process all depth outputs 0.0
Time at depth test 4, 0.1765210628

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


Time to run all tasks in parallel 0.0071260929107666016
Time to process all depth outputs 0.0
Time at depth test 2, 0.007519721984863281
Number of active sets to be considered is 6
Time to run all tasks in parallel 0.006009578704833984
Time to process all depth outputs 0.0
Time at depth test 3, 0.013529300689697266
Number of active sets to be considered is 10
Time to run all tasks in parallel 0.015767574310302734
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 17
Time to run all tasks in parallel 0.01799941062927246
Time to process all depth outputs 0.0
Time at depth test 2, 0.01799941062927246
Number of active sets to be considered is 105
Time to run all tasks in parallel 0.05488228797912598
Time to process all depth outputs 0.0010008811950683594
Time at depth test 3, 0.0738825798034668
Number of active sets to be considered is 478
Time to run all tasks in parallel 0.10472273826599121
Time to process all depth outputs 0.0
Time at depth tes

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


In [28]:
print(f'Gauss Legendre ESF: {gl_esf}')
print(f'Smolyak ESF: {sm_esf}')

Gauss Legendre ESF: 0.12237052154747201
Smolyak ESF: 0.12140091065843066


## Smolyak SF Expression

In [29]:
def compute_sf_exprs_regions_smolyak(
    theta_bounds_list,
    theta_regions_list,
    joint_pdf_expr,
    d_syms,
    theta_syms,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    FIXED VERSION:
    - For each region combo, contributions from Smolyak nodes are GATED by the region constraints
      after substituting theta(theta_prev, d, u_node).
    - This prevents double-counting across region combos (the main bug in the old version).

    Returns:
      sf_exprs:  list of sympy expressions (each is gated to its combo)
      sf_regions: list of region constraint lists (same as before)
    """
    n_theta = len(theta_syms)

    # Sparse grid nodes/weights on [-1,1]^n (Chaospy gives expectation weights)
    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])
    nodes_u, weights_expectation = cp.generate_quadrature(
        order=level,
        dist=dist,
        rule=rule,
        sparse=True,
        growth=growth,
    )
    nodes_u = np.array(nodes_u).T
    weights_expectation = np.array(weights_expectation).flatten()

    # Convert expectation weights to integral weights over [-1,1]^n (volume = 2^n)
    weights_u = (2.0 ** n_theta) * weights_expectation

    # Enumerate region combos (same as your current code)
    region_sizes = [bounds.shape[0] for bounds in theta_bounds_list]
    region_combos = list(itools.product(*[range(r) for r in region_sizes]))

    sf_exprs = []
    sf_regions = []

    for region_combo in region_combos:

        # 1) Build region constraints in (theta_syms, d_syms)
        combo_constraints = []
        for level_idx, region_idx in enumerate(region_combo):
            rows = theta_regions_list[level_idx][region_idx]
            for row in rows:
                t_coeffs = row[:level_idx+1]
                d_coeffs = row[level_idx+1:-1]
                const = row[-1]

                lhs = sum(c * theta_syms[i] for i, c in enumerate(t_coeffs)) + \
                    sum(c * d for c, d in zip(d_coeffs, d_syms)) + const
                ineq = lhs <= 0

                if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                    combo_constraints.append(ineq)

        # (Optional) stable order; not dedupe, but OK
        combo_constraints = sorted(combo_constraints, key=str)

        # 2) Smolyak quadrature expression with node-wise gating
        sf_sum = 0

        for u_vec, w_u in zip(nodes_u, weights_u):

            theta_vals = []
            jacobian = 1

            # Sequentially compute theta_k(u, d) for this combo
            for level_idx, region_idx in enumerate(region_combo):
                bounds = theta_bounds_list[level_idx][region_idx]
                bound_inputs = theta_vals + list(d_syms)

                # IMPORTANT: keep your existing convention here:
                # bounds[0] is t_min, bounds[1] is t_max (do NOT change since GL works for you)
                t_min = affine_expr(bounds[0], bound_inputs)
                t_max = affine_expr(bounds[1], bound_inputs)

                u_k = float(u_vec[level_idx])
                t_k = 0.5 * (t_max - t_min) * u_k + 0.5 * (t_max + t_min)

                theta_vals.append(t_k)
                jacobian *= 0.5 * (t_max - t_min)

            # Substitute theta into PDF (so pdf becomes expression in d_syms)
            theta_subs = {sym: val for sym, val in zip(theta_syms, theta_vals)}
            pdf_val = joint_pdf_expr.subs(theta_subs)

            # GATE this node's contribution by combo feasibility
            if combo_constraints:
                gated_constraints = []
                for ineq in combo_constraints:
                    ineq_sub = ineq.subs(theta_subs)
                    # After substitution this should depend only on d_syms (and constants)
                    if not isinstance(ineq_sub, (BooleanTrue, BooleanFalse)):
                        gated_constraints.append(ineq_sub)

                if gated_constraints:
                    cond = sp.And(*gated_constraints)
                    term = sp.Piecewise(
                        (w_u * jacobian * pdf_val, cond),
                        (0, True)
                    )
                else:
                    # constraints evaluated to True/False already
                    term = w_u * jacobian * pdf_val
            else:
                term = w_u * jacobian * pdf_val

            sf_sum += term

        sf_exprs.append(sp.simplify(sf_sum))
        sf_regions.append(combo_constraints)

    return sf_exprs, sf_regions


def compute_sf_smolyak_symbolic_fast(
    theta_bounds_list,
    theta_regions_list,
    joint_pdf_expr,
    d_syms,
    theta_syms,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    Smolyak symbolic SF (node-wise region selection) with SAFE Piecewise creation.

    Returns:
        sf_expr (sympy Expr), stats (dict)
    """
    n_theta = len(theta_syms)

    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])
    nodes_u, wE = cp.generate_quadrature(
        order=level, dist=dist, rule=rule, sparse=True, growth=growth
    )
    nodes_u = np.asarray(nodes_u, dtype=float).T
    wE = np.asarray(wE, dtype=float).ravel()
    wU = (2.0 ** n_theta) * wE

    def _region_holds(k: int, region_rows, theta_prev_exprs):
        """Return sympy Boolean condition (in d_syms only) for stage k region feasibility."""
        conds = []
        for row in region_rows:
            t_coeffs = row[:k]        # theta_0..theta_{k-1}
            d_coeffs = row[k:-1]      # d0..d_{nd-1}
            const = row[-1]

            lhs = sum(c * theta_prev_exprs[i] for i, c in enumerate(t_coeffs)) \
                + sum(c * d for c, d in zip(d_coeffs, d_syms)) \
                + const

            ineq = sp.Le(lhs, 0)
            if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                conds.append(ineq)

        return sp.And(*conds) if conds else sp.true

    sf_sum = 0

    for u_vec, w_u in zip(nodes_u, wU):
        theta_vals = []   # sympy expressions in d_syms
        jac = 1

        for k in range(n_theta):
            bound_inputs = theta_vals + list(d_syms)

            # Build non-nested Piecewise by collecting (expr, cond) pairs
            min_pairs = []
            max_pairs = []

            n_regions_k = theta_bounds_list[k].shape[0]
            for r_idx in range(n_regions_k):
                # [tmin_coeffs, tmax_coeffs] (keep your convention)
                bounds = theta_bounds_list[k][r_idx]
                region_rows = theta_regions_list[k][r_idx]

                cond = _region_holds(k, region_rows, theta_vals)
                tmin_r = affine_expr(bounds[0], bound_inputs)
                tmax_r = affine_expr(bounds[1], bound_inputs)

                min_pairs.append((tmin_r, cond))
                max_pairs.append((tmax_r, cond))

            # IMPORTANT: add a default branch to avoid Sympy as_set/ITE rewrite errors
            # Use the last expression as fallback. (Assumes regions cover the space; if not, it still prevents crashes.)
            pw_tmin = sp.Piecewise(*min_pairs, (min_pairs[-1][0], True))
            pw_tmax = sp.Piecewise(*max_pairs, (max_pairs[-1][0], True))

            u_k = float(u_vec[k])
            t_k = 0.5 * (pw_tmax - pw_tmin) * u_k + 0.5 * (pw_tmax + pw_tmin)

            theta_vals.append(t_k)
            jac *= 0.5 * (pw_tmax - pw_tmin)

        pdf_val = joint_pdf_expr.subs(
            {sym: val for sym, val in zip(theta_syms, theta_vals)})
        sf_sum += w_u * jac * pdf_val

    stats = {"n_nodes": int(nodes_u.shape[0])}
    # Avoid simplify() here; it can take forever on Piecewise-heavy expressions
    return sf_sum, stats

## Gaussian Legendre SF Expressions

In [30]:
def compute_sf_exprs_regions(
    theta_bounds_list,
    theta_regions_list,
    joint_pdf_expr,
    d_syms,
    n_gl_list,
    theta_syms
):
    n_theta = len(theta_syms)
    quad_data = [np.polynomial.legendre.leggauss(n) for n in n_gl_list]
    region_sizes = [bounds.shape[0] for bounds in theta_bounds_list]
    region_combos = generate_region_combos(region_sizes, n_gl_list)

    sf_exprs = []
    sf_regions = []

    for region_combo in region_combos:
        combo_ptr = 0
        # Initialize integration paths: (theta_vals, weight, scale, constraints)
        paths = [([], 1, 1, [])]

        for level in range(n_theta):
            xi, wi = quad_data[level]
            new_paths = []

            for theta_vals, weight, scale, constraints in paths:
                region_idx = region_combo[combo_ptr]
                combo_ptr += 1

                bound_inputs = theta_vals + list(d_syms)
                bounds = theta_bounds_list[level][region_idx]
                t_min = affine_expr(bounds[0], bound_inputs)
                t_max = affine_expr(bounds[1], bound_inputs)

                # Get level-specific region constraints
                rows = theta_regions_list[level][region_idx]
                level_constraints = []
                for row in rows:
                    t_coeffs = row[:level]
                    d_coeffs = row[level:-1]
                    const = row[-1]
                    lhs = sum(c * theta_vals[i] for i, c in enumerate(t_coeffs)) + \
                        sum(c * d for c, d in zip(d_coeffs, d_syms)) + const
                    ineq = lhs <= 0
                    # level_constraints.append(sp.simplify(lhs <= 0))
                    if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                        level_constraints.append(ineq)

                new_constraints = constraints + level_constraints

                # Quadrature expansion for this level
                for q in range(len(xi)):
                    t = 0.5 * (t_max - t_min) * xi[q] + 0.5 * (t_max + t_min)
                    # new_theta_vals = theta_vals + [sp.simplify(t)]
                    new_theta_vals = theta_vals + [t]
                    new_weight = weight * wi[q]
                    new_scale = scale * 0.5 * (t_max - t_min)
                    new_paths.append(
                        (new_theta_vals, new_weight, new_scale, new_constraints))

            paths = new_paths

        # Final integration and region collection
        sf_sum = 0
        all_constraints = []
        for theta_vals, weight, scale, constraints in paths:
            theta_subs = {sym: val for sym, val in zip(theta_syms, theta_vals)}
            pdf_val = joint_pdf_expr.subs(theta_subs)
            sf_sum += weight * scale * pdf_val
            all_constraints.extend(constraints)

        # Deduplicate constraints symbolically
        unique_constraints = []
        for c in all_constraints:
            if isinstance(c, (BooleanTrue, BooleanFalse)):
                print(f'Skipping trivial constraint: {c}')
            if not any(normalized_lhs(c) == normalized_lhs(u) and type(c) == type(u) for u in unique_constraints if normalized_lhs(u) is not None):
                unique_constraints.append(c)

        sf_exprs.append(sf_sum)
        # sf_regions.append(sorted(all_constraints, key=str))
        # sf_exprs.append(sp.simplify(sf_sum))
        sf_regions.append(sorted(unique_constraints, key=str))

    return sf_exprs, sf_regions

## Pyomo Model

In [31]:
def preprocess_sf_expressions(
    expr_list: List[sp.Expr],
    region_list: List[List[sp.Expr]],
) -> Tuple[List[Callable], List[List[Tuple[str, Callable, Callable]]], List[str]]:
    """
    Preprocess symbolic SF and region expressions into callable Pyomo functions.

    Returns:
        sf_pyomo_funcs:   list of callables f(var_dict)->Pyomo expression
        region_pyomo_funcs: list (per SF expr) of [(rel_op, lhs_func, rhs_func), ...]
        var_names:        sorted list of variable names used
    """
    assert len(expr_list) == len(
        region_list), "Mismatch between expressions and regions"

    sf_pyomo_funcs = []
    region_pyomo_funcs = []
    all_syms = set()

    # collect symbols + validate region expressions
    for sf_expr, reg_exprs in zip(expr_list, region_list):
        all_syms.update(sf_expr.free_symbols)
        for reg_expr in reg_exprs:
            if not isinstance(reg_expr, SympyRelational):
                raise ValueError(
                    f"Region expression {reg_expr} is not a Sympy relational (<=,>=,==,...)"
                )
            all_syms.update(reg_expr.free_symbols)

    var_names = sorted(str(s) for s in all_syms)

    # SF expression callables (Pyomo-safe exp/log/pi)
    for sf_expr in expr_list:
        sf_func = sp.lambdify(
            var_names,
            sf_expr,
            modules=[{"exp": pyo.exp, "log": pyo.log,
                      "ln": pyo.log, "pi": math.pi}, "sympy"],
        )
        sf_pyomo_funcs.append(lambda var_dict, f=sf_func: f(
            *[var_dict[v] for v in var_names]))

    # Region constraint callables (lhs/rhs algebra only)
    for reg_exprs in region_list:
        region_funcs = []
        for reg in reg_exprs:
            lhs_func = sp.lambdify(var_names, reg.lhs, modules="sympy")
            rhs_func = sp.lambdify(var_names, reg.rhs, modules="sympy")
            region_funcs.append((reg.rel_op, lhs_func, rhs_func))
        region_pyomo_funcs.append(region_funcs)

    return sf_pyomo_funcs, region_pyomo_funcs, var_names


def embed_sf_constraints_to_model(
    instance: ConcreteModel,
    sf_pyomo_funcs: List[Callable],
    region_pyomo_funcs: List[List[Tuple[str, Callable, Callable]]],
    var_names: List[str],
    bounds_dict: Dict[str, Tuple[float, float]],
    big_m: float = 1e4,
    initial_target: float = 1.0,
):
    """
    Embed SF constraints and region logic into a Pyomo model using preprocessed functions.
    """
    if not hasattr(instance, 'generated_constraints'):
        instance.generated_constraints = ConstraintList()

    if not hasattr(instance, 'region_constraints'):
        instance.region_constraints = ConstraintList()

    if not hasattr(instance, 'region_binaries'):
        instance.region_binaries = Var(
            RangeSet(len(sf_pyomo_funcs)), within=Binary)

    # Create design variables

    for var_name in var_names:
        if not hasattr(instance, var_name):
            setattr(instance, var_name, Var(bounds=bounds_dict[var_name]))

    # Create mapping for lambdified expressions
    var_dict = {name: getattr(instance, name) for name in var_names}

    # Mutable Param for target
    if not hasattr(instance, 'sf_target'):
        instance.sf_target = Param(mutable=True, initialize=initial_target)

    # Create sf_var and constraint
    if not hasattr(instance, 'sf'):
        instance.sf = Var(within=NonNegativeReals)

    # Store SF expressions
    sf_expr_pyomo_list = []
    for i, sf_func in enumerate(sf_pyomo_funcs):
        sf_expr = sf_func(var_dict)
        sf_expr_pyomo_list.append(sf_expr)
        # Big-M constraint: sf_expr >= target - M(1 - δ)
        delta = instance.region_binaries[i + 1]
        instance.generated_constraints.add(
            sf_expr >= instance.sf_target - big_m * (1 - delta))

    # Link to overall sf var
    if not hasattr(instance, 'sf_con'):
        instance.sf_con = Constraint(expr=instance.sf == sum(
            sf_expr_pyomo_list[i] * instance.region_binaries[i + 1] for i in range(len(sf_expr_pyomo_list))))

    # Region constraints
    for i, region_funcs in enumerate(region_pyomo_funcs):
        delta = instance.region_binaries[i + 1]

        for rel_op, lhs_func, rhs_func in region_funcs:
            lhs = lhs_func(*[var_dict[v] for v in var_names])
            rhs = rhs_func(*[var_dict[v] for v in var_names])

            if rel_op == '<=':
                instance.region_constraints.add(
                    lhs <= rhs + big_m * (1 - delta))

            elif rel_op == '<':
                instance.region_constraints.add(
                    lhs <= rhs - 1e-6 + big_m * (1 - delta))

            elif rel_op == '>=':
                instance.region_constraints.add(
                    lhs >= rhs - big_m * (1 - delta))

            elif rel_op == '>':
                instance.region_constraints.add(
                    lhs >= rhs + 1e-6 - big_m * (1 - delta))

            elif rel_op == '==':
                instance.region_constraints.add(
                    lhs >= rhs - big_m * (1 - delta))
                instance.region_constraints.add(
                    lhs <= rhs + big_m * (1 - delta))

            else:
                raise NotImplementedError(f"Unsupported operator: {rel_op}")

    # Exclusivity
    instance.region_exclusivity = Constraint(expr=sum(
        instance.region_binaries[i + 1] for i in range(len(sf_pyomo_funcs))) == 1)

In [32]:
import pickle
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
import sympy as sp


def save_sf_cache(
    filepath: str,
    *,
    sf_exprs: List[sp.Expr],
    sf_regions: List[List[sp.Expr]],
    metadata: Optional[Dict[str, Any]] = None,
):
    payload = {
        "sf_exprs": sf_exprs,
        "sf_regions": sf_regions,
        "metadata": metadata or {},
    }

    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"[cache] saved -> {path.resolve()}")


def load_sf_cache(filepath: str):
    with open(filepath, "rb") as f:
        payload = pickle.load(f)
    return payload["sf_exprs"], payload["sf_regions"], payload.get("metadata", {})


def cache_key_gl(nt: int, nd: int, n_gl_list: List[int]) -> str:
    return f"gl_nt{nt}_nd{nd}_nq{'-'.join(map(str, n_gl_list))}.pkl"


def cache_key_smolyak(nt: int, nd: int, level: int, rule: str, growth: bool) -> str:
    g = "growth1" if growth else "growth0"
    return f"smolyak_nt{nt}_nd{nd}_L{level}_{rule}_{g}.pkl"

In [33]:
# theta_bounds_list, theta_regions_list = get_bounds_regions(sols=sol_list)

# theta_syms = sp.symbols(f'theta_0:{nt}')
# d_syms = sp.symbols(f'd0:{nd}')
# theta_0, theta_1 = theta_syms
# # d1, d2 = d_syms
# d1, d2, d3 = d_syms

# # Reduce n_gl_list [1,1]
# n_gl_list = [6, 6]

# # joint_pdf_expr = (2/sp.pi) * sp.exp(-2 *
# #                                     ((theta_0 - 2) ** 2 + (theta_1 - 2) ** 2))
# joint_pdf_expr = (1/(1.2 * sp.pi * (theta_0 - 8))) * sp.exp(
#     -1.39*(sp.ln(theta_0 - 8))**2 - 0.5*(theta_1 - 7)**2
# )

# start_gl = time.perf_counter()
# sf_exprs_gl, sf_regions_gl = compute_sf_exprs_regions(
#     theta_bounds_list=theta_bounds_list,
#     theta_regions_list=theta_regions_list,
#     joint_pdf_expr=joint_pdf_expr,
#     d_syms=d_syms,
#     n_gl_list=n_gl_list,
#     theta_syms=theta_syms,
# )
# end_gl = time.perf_counter()
# time_gl = end_gl - start_gl
# print(f"Gauss–Legendre symbolic SF build time: {time_gl:.4f} s")

In [34]:
# # save
# path = "cache/" + cache_key_gl(nt=nt, nd=nd, n_gl_list=n_gl_list)
# save_sf_cache(
#     path,
#     sf_exprs=sf_exprs_gl,
#     sf_regions=sf_regions_gl,
#     metadata={
#         "method": "gauss_legendre",
#         "nt": nt, "nd": nd,
#         "n_gl_list": n_gl_list,
#         "pdf": "pdf_expr",
#     },
# )

In [35]:
# # I commented this
# path = "cache/" + cache_key_gl(nt=nt, nd=nd, n_gl_list=n_gl_list)
# sf_exprs_gl, sf_regions_gl, meta = load_sf_cache(path)

In [36]:
# print(f'Number of Gaussian Legendre SF expressions : {len(sf_exprs_gl)}')
# print(f'Number of Gaussian Legendre critical regions: {len(sf_regions_gl)}')

In [37]:
# # --- Smolyak symbolic SF ---
# theta_bounds_list, theta_regions_list = get_bounds_regions(sols=sol_list)
# theta_syms = sp.symbols(f'theta_0:{nt}')
# d_syms = sp.symbols(f'd0:{nd}')
# theta_0, theta_1 = theta_syms
# # d1, d2 = d_syms
# d1, d2, d3 = d_syms
# # joint_pdf_expr = (2/sp.pi) * sp.exp(-2 *
# #                                     ((theta_0 - 2) ** 2 + (theta_1 - 2) ** 2))
# joint_pdf_expr = (1/(1.2 * sp.pi * (theta_0 - 8))) * sp.exp(
#     -1.39*(sp.ln(theta_0 - 8))**2 - 0.5*(theta_1 - 7)**2)
# 
# smolyak_level = 6      # or whatever level you want
# smolyak_rule = "gaussian"  # or "clenshaw_curtis", "gaussian"
# 
# start_sm = time.perf_counter()
# sf_expr_sm, sm_stats = compute_sf_smolyak_symbolic_fast(
#     theta_bounds_list, theta_regions_list, joint_pdf_expr, d_syms, theta_syms,
#     level=smolyak_level, rule=smolyak_rule, growth=True
# )
# end_sm = time.perf_counter()
# time_sm = end_sm - start_sm
# print(f"Smolyak symbolic SF build time:       {time_sm:.4f} s")

In [38]:
# # Optional: simple speedup metric
# if time_sm > 0:
#     print(f"Smolyak / Gauss–Legendre time ratio: {time_sm/time_gl:.3f}")
# 
# print(f'Number of Smolyak SF expressions : {len([sf_expr_sm])}')

In [39]:
# design_bounds = {f'd{i}': bounds for i, bounds in enumerate(d_bounds)}
# 
# sf_funcs_gl, region_funcs_gl, var_names_gl = preprocess_sf_expressions(
#     expr_list=sf_exprs_gl, region_list=sf_regions_gl)
# m_sf_gl = ConcreteModel()
# embed_sf_constraints_to_model(instance=m_sf_gl, sf_pyomo_funcs=sf_funcs_gl,
#                               region_pyomo_funcs=region_funcs_gl, var_names=var_names_gl, bounds_dict=design_bounds)
# # m_sf_gl.obj = Objective(expr=10*m_sf_gl.d0 - 10*m_sf_gl.d1, sense=minimize)
# m_sf_gl.obj = Objective(expr=10*m_sf_gl.d0 + 3 *
#                         m_sf_gl.d1 + 10*m_sf_gl.d2, sense=minimize)

In [40]:
# sf_target = 0.4
# m_sf_gl.sf_target.set_value(sf_target)
# results = SolverFactory('gams', solver='baron').solve(m_sf_gl, tee=True)

In [41]:
def smolyak_nodes_weights(
    n_theta: int,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    Returns:
        nodes_u: (N, n_theta) array of Smolyak nodes in u-space (each u_k in [-1,1])
        weights_u: (N,) array of weights for integrating over [-1,1]^n_theta
                   i.e., sum_i weights_u[i] * f(nodes_u[i]) ≈ ∫_{[-1,1]^n} f(u) du
    """
    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    # Chaospy returns nodes shape (n_theta, N) and weights for expectation
    nodes, wE = cp.generate_quadrature(
        order=level,
        dist=dist,
        rule=rule,
        sparse=True,
        growth=growth,
    )

    nodes = np.asarray(nodes, dtype=float)        # (n_theta, N)
    wE = np.asarray(wE, dtype=float).ravel()      # (N,)

    nodes_u = nodes.T                             # (N, n_theta)

    # Convert expectation weights to integral weights over [-1,1]^n:
    # E[f(U)] = ∫ f(u) p(u) du with p(u)=1/2^n on [-1,1]^n
    # => ∫ f(u) du = 2^n * E[f(U)]
    weights_u = (2.0 ** n_theta) * wE

    return nodes_u, weights_u


def build_exact_smolyak_sf_minlp(
    *,
    # from your get_bounds_regions()
    # list over k: array shape (Rk, 2, dim_k+nd+1) coefficients for [tmin,tmax]
    theta_bounds_list,
    # list over k: list/array of regions; each region is 2D array rows [coeffs..., const]
    theta_regions_list,
    # smolyak nodes/weights
    nodes_u,               # (N, nt) numeric
    weights_u,             # (N,) numeric
    # design variables
    d_bounds,              # list of (lb,ub) for d0..d{nd-1}
    t_bounds,
    sf_target=0.4,
    big_m=1e4,
    # sympy expression in theta_syms (optional, for later use if needed
    pdf_expr=None,
):
    """
    Exact Smolyak SF(d) embedding as MINLP with binaries.

    Assumptions consistent with your code:
      - theta_bounds_list[k][r][0] and [1] are affine coeff vectors for tmin/tmax in terms of [theta_0..theta_{k-1}, d0..d{nd-1}, 1]
      - theta_regions_list[k][r] contains rows encoding region inequalities in the same variable order:
            sum_i a_i * theta_i + sum_j b_j * d_j + const <= 0
        where theta_i ranges i=0..k (or 0..k-1 depending on your region definition);
        We enforce them using the available theta variables up to k-1 (sequential construction).
    """
    N = int(nodes_u.shape[0])
    nt = int(nodes_u.shape[1])
    nd = len(d_bounds)

    m = pyo.ConcreteModel()

    # --- sets ---
    m.N = pyo.RangeSet(0, N-1)
    m.K = pyo.RangeSet(0, nt-1)

    # per-k regions
    Rk = [theta_bounds_list[k].shape[0] for k in range(nt)]
    m.R = pyo.Set(m.K, initialize=lambda mm, k: list(range(Rk[int(k)])))

    # --- design vars ---
    for j, (lb, ub) in enumerate(d_bounds):
        setattr(m, f"d{j}", pyo.Var(bounds=(lb, ub), initialize=0.5*(lb+ub)))

    def dvar(j):
        return getattr(m, f"d{int(j)}")

    # --- node-wise theta, bounds, jacobian ---
    m.theta = pyo.Var(m.K, m.N,
                      bounds=lambda mm, k, n: t_bounds[int(k)],
                      initialize=lambda mm, k, n: 0.5*(t_bounds[int(k)][0] + t_bounds[int(k)][1]))

    m.tmin = pyo.Var(m.K, m.N,
                     bounds=lambda mm, k, n: t_bounds[int(k)])

    m.tmax = pyo.Var(m.K, m.N,
                     bounds=lambda mm, k, n: t_bounds[int(k)])
    m.jac = pyo.Var(m.N, within=pyo.NonNegativeReals, initialize=1.0)

    # We can’t directly index m.z by (k,r,n) unless we define a 3D index set:
    m.KRN = pyo.Set(dimen=3,
                    initialize=[(k, r, n) for k in range(nt)
                                for n in range(N) for r in range(Rk[k])]
                    )
    m.z = pyo.Var(m.KRN, within=pyo.Binary)

    # --- helper: affine evaluation for bound coeff vector ---
    # coeffs format: [theta_0..theta_{k-1}, d0..d{nd-1}, 1]
    def affine_bound_expr(k, n, coeff_vec):
        k = int(k)
        n = int(n)
        coeff_vec = np.asarray(coeff_vec, dtype=float).ravel()
        # expected length = (k) + nd + 1  where k = number of previous thetas
        expr = 0
        # thetas used are theta[0..k-1]
        for i in range(k):
            expr += float(coeff_vec[i]) * m.theta[i, n]
        # d's
        offset = k
        for j in range(nd):
            expr += float(coeff_vec[offset + j]) * dvar(j)
        # constant
        expr += float(coeff_vec[-1])
        return expr

    # --- 1) region exclusivity for each (k,n) ---
    m.region_one = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            m.region_one.add(
                sum(m.z[(k, r, n)] for r in range(Rk[k])) == 1
            )

    # --- 2) define tmin/tmax using selected region bounds (big-M equality) ---
    m.bound_link = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            for r in range(Rk[k]):
                # coefficients for this region r at level k
                # stored as [tmin_coeffs, tmax_coeffs] (your get_bounds_regions uses [min,max] ordering)
                tmin_coeff = theta_bounds_list[k][r][0]
                tmax_coeff = theta_bounds_list[k][r][1]

                tmin_expr = affine_bound_expr(k, n, tmin_coeff)
                tmax_expr = affine_bound_expr(k, n, tmax_coeff)

                zkrn = m.z[(k, r, n)]
                # enforce equality when active, relaxed when inactive
                m.bound_link.add(m.tmin[k, n] - tmin_expr <= big_m*(1 - zkrn))
                m.bound_link.add(m.tmin[k, n] - tmin_expr >= -big_m*(1 - zkrn))
                m.bound_link.add(m.tmax[k, n] - tmax_expr <= big_m*(1 - zkrn))
                m.bound_link.add(m.tmax[k, n] - tmax_expr >= -big_m*(1 - zkrn))

    # --- 3) region membership constraints (linear) with big-M ---
    # theta_regions_list[k][r] is an array of rows representing inequalities.
    # Each row: [theta_coeffs..., d_coeffs..., const] so that lhs <= 0
    m.region_ineq = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            for r in range(Rk[k]):
                zkrn = m.z[(k, r, n)]
                rows = theta_regions_list[k][r]
                for row in rows:
                    row = np.asarray(row, dtype=float).ravel()
                    # interpret as: sum_{i<=k?} a_i*theta_i + sum_j b_j*d_j + const <= 0
                    # We only have theta[0..k-1] available when constructing bounds for theta_k,
                    # so we will use min(len(theta_coeffs), k) terms.
                    # (This matches your sequential construction usage.)
                    # Determine split: first (k) entries are theta coeffs for theta_0..theta_{k-1}
                    # next nd are d coeffs, last is const
                    # If your stored rows include theta_k too, set theta_count = k+1 instead.
                    theta_count = k  # sequential (uses previous theta only)
                    if len(row) != theta_count + nd + 1:
                        # If your data includes theta_k as well, switch automatically:
                        if len(row) == (k+1) + nd + 1:
                            theta_count = k+1
                        else:
                            raise ValueError(
                                f"Bad region row length at k={k}, r={r}. Got {len(row)}.")

                    lhs = 0
                    for i in range(theta_count):
                        lhs += float(row[i]) * m.theta[i, n]
                    for j in range(nd):
                        lhs += float(row[theta_count + j]) * dvar(j)
                    lhs += float(row[-1])

                    # enforce lhs <= 0 when active
                    m.region_ineq.add(lhs <= big_m*(1 - zkrn))

    # --- 4) theta mapping from u to [tmin,tmax] ---
    m.theta_map = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            ukn = float(nodes_u[n, k])
            m.theta_map.add(
                m.theta[k, n] == 0.5*(m.tmax[k, n] - m.tmin[k, n]) *
                ukn + 0.5*(m.tmax[k, n] + m.tmin[k, n])
            )
            # also enforce tmax >= tmin (helps numerics)
            m.theta_map.add(m.tmax[k, n] >= m.tmin[k, n] + 1e-8)

    # --- 5) jacobian product ---
    # jac[n] = Π_k 0.5*(tmax - tmin)
    m.jac_def = pyo.ConstraintList()
    for n in range(N):
        expr = 1
        for k in range(nt):
            expr *= 0.5*(m.tmax[k, n] - m.tmin[k, n])
        m.jac_def.add(m.jac[n] == expr)

    # --- 6) SF expression + target constraint ---
    # pdf(theta) = 1/(1.2*pi*(theta0-8)) * exp(-1.39*(log(theta0-8))^2 - 0.5*(theta1-7)^2)
    # Need theta0 > 8 strictly. Put a small epsilon lower bound via constraint.
    eps = 1e-6

    m.sf_target = pyo.Param(mutable=True, initialize=float(sf_target))
    m.SF = pyo.Expression(
        expr=sum(float(weights_u[n]) * m.jac[n] * pdf_expr(m, n) for n in range(N)))
    m.sf_con = pyo.Constraint(expr=m.SF >= m.sf_target)

    return m

# def warmstart_exact_smolyak_minlp(
#     m: pyo.ConcreteModel,
#     *,
#     theta0_init: float = 9.0,    # must be > 8 (log domain)
#     theta1_init: float = 7.0,
#     tmin_init: float = 8.5,      # must be > 8
#     tmax_init: float = 9.5,      # must be > tmin_init
# ):
#     """
#     Warm-start the Smolyak MINLP so BARON can safely evaluate exp/log terms.
# 
#     This:
#       - sets design variables to midpoints
#       - initializes theta, tmin, tmax consistently
#       - initializes jacobian > 0
#       - satisfies region exclusivity by picking r=0 everywhere
# 
#     It does NOT guarantee feasibility of region constraints.
#     """
# 
#     # ------------------------
#     # 1) Design variables d_j
#     # ------------------------
#     j = 0
#     while hasattr(m, f"d{j}"):
#         v = getattr(m, f"d{j}")
#         lb, ub = v.bounds
#         v.set_value(0.5 * (lb + ub))
#         j += 1
# 
#     # ------------------------
#     # 2) Theta, tmin, tmax
#     # ------------------------
#     for n in m.N:
#         # theta values
#         if (0, n) in m.theta:
#             m.theta[0, n].set_value(theta0_init)
#         if (1, n) in m.theta:
#             m.theta[1, n].set_value(theta1_init)
# 
#         # bounds
#         for k in m.K:
#             if (k, n) in m.tmin:
#                 m.tmin[k, n].set_value(tmin_init)
#             if (k, n) in m.tmax:
#                 m.tmax[k, n].set_value(tmax_init)
# 
#         # jacobian must be positive
#         if n in m.jac:
#             m.jac[n].set_value(
#                 max(1e-3, (0.5 * (tmax_init - tmin_init)) ** len(list(m.K)))
#             )
# 
#     # ------------------------
#     # 3) Region binaries
#     # ------------------------
#     # Pick region r=0 for every (k,n) → satisfies sum_r z = 1
#     if hasattr(m, "KRN") and hasattr(m, "z"):
#         for (k, r, n) in m.KRN:
#             m.z[(k, r, n)].set_value(1 if r == 0 else 0)

In [42]:
# # Build nodes/weights once
# nodes_u, weights_u = smolyak_nodes_weights(
#     n_theta=2, level=8, rule="gaussian", growth=True)

# # Your bounds/regions from ppopt solutions:
# theta_bounds_list, theta_regions_list = get_bounds_regions(sols=sol_list)

# def pdf_expr(m, n):
#     eps = 1e-6
#     S = m.theta[0, n]
#     D = m.theta[1, n]
#     x = S - 8.0 + eps
#     return (1.0/(1.2*math.pi*x))*pyo.exp(-1.39*(pyo.log(x))**2 - 0.5*(D - 7.0)**2)

# # def pdf_expr(m, n):
# #     t0 = m.theta[0,n]
# #     t1 = m.theta[1,n]
# #     return (2/math.pi) * pyo.exp(-2*((t0-2)**2 + (t1-2)**2))


# m_sf_sm_exact = build_exact_smolyak_sf_minlp(
#     theta_bounds_list=theta_bounds_list,
#     theta_regions_list=theta_regions_list,
#     nodes_u=nodes_u,
#     weights_u=weights_u,
#     d_bounds=d_bounds,
#     t_bounds= t_bounds,
#     sf_target=0.4,
#     big_m=1e5,
#     pdf_expr=pdf_expr,
# )

# # Your objective (same as before)
# # m_sf_sm_exact.obj = pyo.Objective(
# #     expr=10*m_sf_sm_exact.d0 - 10*m_sf_sm_exact.d1, sense=pyo.minimize)
# m_sf_sm_exact.obj = pyo.Objective(
#     expr=10*m_sf_sm_exact.d0 + 3*m_sf_sm_exact.d1 + 10*m_sf_sm_exact.d2, sense=pyo.minimize)

# warmstart_exact_smolyak_minlp(m_sf_sm_exact)


# res = pyo.SolverFactory("gams", solver="baron").solve(
#     m_sf_sm_exact, tee=True, keepfiles=True, tmpdir=r"C:\gams_tmp"
# )

In [43]:
def build_base_model_esf(d_bounds):
    m = pyo.ConcreteModel()

    # Design variables
    for j, (lb, ub) in enumerate(d_bounds):
        setattr(m, f"d{j}", pyo.Var(bounds=(lb, ub), initialize=0.5*(lb + ub)))

    def dvar(j):
        return getattr(m, f"d{int(j)}")
    m.dvar = dvar  # helper for easier access in pdf_expr

    # global ESF (sum of prob-weighted block SFs)
    m.ESF = pyo.Expression(initialize=0.0)

    # store number of design vars
    m.nd = len(d_bounds)

    return m


def add_sf_block(
    m,
    name,
    theta_bounds_list,
    theta_regions_list,
    nodes_u,
    weights_u,
    t_bounds,
    pdf_expr,
    probability,
    big_m=1e5,
):
    N = int(nodes_u.shape[0])
    nt = len(theta_bounds_list)
    nd = m.nd

    block = pyo.Block()
    setattr(m, name, block)
    block = getattr(m, name)

    block.probability = float(probability)
    block.N = pyo.RangeSet(0, N - 1)
    block.K = pyo.RangeSet(0, nt - 1)
    Rk = [theta_bounds_list[k].shape[0] for k in range(nt)]
    block.R = pyo.Set(block.K, initialize=lambda b, k: list(range(Rk[int(k)])))

    # nodes/weights
    block.nodes_u = pyo.Param(
        block.N, block.K,
        initialize=lambda b, n, k: float(nodes_u[n, k]),
        mutable=False,
    )
    block.weights_u = pyo.Param(
        block.N,
        initialize=lambda b, n: float(weights_u[n]),
        mutable=False,
    )

    # theta, tmin, tmax, jac
    block.theta = pyo.Var(
        block.K, block.N,
        bounds=lambda b, k, n: t_bounds[int(k)],
        initialize=lambda b, k, n: 0.5 *
        (t_bounds[int(k)][0] + t_bounds[int(k)][1]),
    )
    block.tmin = pyo.Var(
        block.K, block.N,
        bounds=lambda b, k, n: t_bounds[int(k)],
    )
    block.tmax = pyo.Var(
        block.K, block.N,
        bounds=lambda b, k, n: t_bounds[int(k)],
    )
    block.jac = pyo.Var(block.N, within=pyo.NonNegativeReals, initialize=1.0)

    block.KRN = pyo.Set(
        dimen=3,
        initialize=[(k, r, n)
                    for k in range(nt)
                    for n in range(N)
                    for r in range(Rk[k])])
    block.z = pyo.Var(block.KRN, within=pyo.Binary)

    # affine helper
    def affine_bound_expr(b, k, n, coeff_vec):
        k = int(k)
        n = int(n)
        coeff_vec = np.asarray(coeff_vec, dtype=float).ravel()
        expr = 0
        for i in range(k):
            expr += float(coeff_vec[i]) * block.theta[i, n]
        offset = k
        for j in range(nd):
            expr += float(coeff_vec[offset + j]) * m.dvar(j)
        expr += float(coeff_vec[-1])
        return expr
    block._affine_bound_expr = affine_bound_expr

    # region exclusivity
    block.region_one = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            block.region_one.add(
                sum(block.z[(k, r, n)] for r in range(Rk[k])) == 1
            )

    # tmin/tmax big-M
    block.bound_link = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            for r in range(Rk[k]):
                tmin_coeff = theta_bounds_list[k][r][0]
                tmax_coeff = theta_bounds_list[k][r][1]

                tmin_expr = affine_bound_expr(block, k, n, tmin_coeff)
                tmax_expr = affine_bound_expr(block, k, n, tmax_coeff)

                zkrn = block.z[(k, r, n)]
                block.bound_link.add(
                    block.tmin[k, n] - tmin_expr <= big_m*(1 - zkrn))
                block.bound_link.add(
                    block.tmin[k, n] - tmin_expr >= -big_m*(1 - zkrn))
                block.bound_link.add(
                    block.tmax[k, n] - tmax_expr <= big_m*(1 - zkrn))
                block.bound_link.add(
                    block.tmax[k, n] - tmax_expr >= -big_m*(1 - zkrn))

    # region membership
    block.region_ineq = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            for r in range(Rk[k]):
                zkrn = block.z[(k, r, n)]
                rows = theta_regions_list[k][r]
                for row in rows:
                    row = np.asarray(row, dtype=float).ravel()
                    theta_count = k
                    if len(row) == (k + 1) + nd + 1:
                        theta_count = k + 1
                    elif len(row) != theta_count + nd + 1:
                        raise ValueError(
                            f"Bad region row length at k={k}, r={r}. Got {len(row)}."
                        )
                    lhs = 0
                    for i in range(theta_count):
                        lhs += float(row[i]) * block.theta[i, n]
                    for j in range(nd):
                        lhs += float(row[theta_count + j]) * m.dvar(j)
                    lhs += float(row[-1])

                    block.region_ineq.add(lhs <= big_m*(1 - zkrn))

    # theta mapping
    block.theta_map = pyo.ConstraintList()
    for k in range(nt):
        for n in range(N):
            ukn = block.nodes_u[n, k]
            block.theta_map.add(
                block.theta[k, n] == 0.5 *
                (block.tmax[k, n] - block.tmin[k, n]) * ukn
                + 0.5*(block.tmax[k, n] + block.tmin[k, n])
            )
            block.theta_map.add(block.tmax[k, n] >= block.tmin[k, n] + 1e-8)

    # jacobian
    block.jac_def = pyo.ConstraintList()
    for n in range(N):
        expr = 1
        for k in range(nt):
            expr *= 0.5*(block.tmax[k, n] - block.tmin[k, n])
        block.jac_def.add(block.jac[n] == expr)

    # SF for this block
    block.sf_expr = pyo.Expression(
        expr=sum(float(block.weights_u[n]) * block.jac[n] * pdf_expr(block, n)
                 for n in range(N))
    )

    # accumulate into global ESF with probability
    if m.ESF.expr is None or m.ESF.is_fixed():
        m.ESF.set_value(block.probability * block.sf_expr)
    else:
        m.ESF.set_value(m.ESF.expr + block.probability * block.sf_expr)

    return block


def set_esf_target(m, target):

    # if global target already exists, just update it
    if hasattr(m, "esf_target") and hasattr(m, "esf_con"):
        m.esf_target.set_value(float(target))
        return m

    # otherwise create Param + Constraint for global ESF
    esf_expr = m.ESF
    m.esf_target = pyo.Param(mutable=True, initialize=float(target))
    m.esf_con = pyo.Constraint(expr=esf_expr >= m.esf_target)
    return m


def _is_sf_block(blk):
    """Heuristic: check that this block has the SF structure."""
    needed_attrs = ["N", "K", "theta", "tmin", "tmax", "jac"]
    for a in needed_attrs:
        if not hasattr(blk, a):
            return False
    # binaries & region set are optional but usually present
    if hasattr(blk, "KRN") and hasattr(blk, "z"):
        return True
    return True  # relax if you sometimes omit z/KRN


def warmstart_exact_smolyak_model(
    m: pyo.ConcreteModel,
    *,
    theta0_init: float = 9.0,
    theta1_init: float = 7.0,
    tmin_init: float = 8.5,
    tmax_init: float = 9.5,
    d_init: list | None = None,
    jac_floor: float = 1e-6,
    prefer_existing: bool = True,
):
    # ---------- global d_j ----------
    j = 0
    while hasattr(m, f"d{j}"):
        v = getattr(m, f"d{j}")
        if d_init is not None and j < len(d_init) and d_init[j] is not None:
            v.set_value(d_init[j])
        else:
            if prefer_existing and (v.value is not None):
                pass
            else:
                lb, ub = v.bounds
                if lb is None and ub is None:
                    v.set_value(0.0)
                elif lb is None:
                    v.set_value(0.5 * ub)
                elif ub is None:
                    v.set_value(0.5 * lb)
                else:
                    v.set_value(0.5 * (lb + ub))
        j += 1

    # ---------- SF blocks ----------
    for blk in m.component_objects(pyo.Block, active=True, descend_into=False):
        # blk is a ScalarBlock (a component); get its data via blk (scalar) or blk[:] (indexed)
        blk_data = blk  # scalar case

        if not _is_sf_block(blk_data):
            continue  # skip non-SF blocks

        # 1) theta, tmin, tmax, jac
        for n in blk_data.N:
            if (0, n) in blk_data.theta:
                if not (prefer_existing and blk_data.theta[0, n].value is not None):
                    blk_data.theta[0, n].set_value(theta0_init)
            if (1, n) in blk_data.theta:
                if not (prefer_existing and blk_data.theta[1, n].value is not None):
                    blk_data.theta[1, n].set_value(theta1_init)

            for k in blk_data.K:
                if (k, n) in blk_data.tmin:
                    v = blk_data.tmin[k, n]
                    if not (prefer_existing and v.value is not None):
                        v.set_value(tmin_init)
                if (k, n) in blk_data.tmax:
                    v = blk_data.tmax[k, n]
                    if not (prefer_existing and v.value is not None):
                        v.set_value(max(tmax_init, tmin_init + 1e-2))

            if n in blk_data.jac:
                v = blk_data.jac[n]
                if not (prefer_existing and v.value is not None):
                    dim = len(list(blk_data.K))
                    base = 0.5 * (tmax_init - tmin_init)
                    v.set_value(max(jac_floor, base**dim))

        # 2) region binaries, if present
        if hasattr(blk_data, "KRN") and hasattr(blk_data, "z"):
            kn_seen = set()
            # preserve any existing values if prefer_existing
            for (k, r, n) in blk_data.KRN:
                zv = blk_data.z[(k, r, n)]
                if prefer_existing and (zv.value is not None):
                    kn_seen.add((k, n))

            # fill in any missing (k, n) with a simple one-hot pattern
            for (k, r, n) in blk_data.KRN:
                if (k, n) in kn_seen:
                    continue
                # for this (k,n), set r=0 as active, others 0
                Rs = [rr for (kk, rr, nn) in blk_data.KRN if kk == k and nn == n]
                for rr in Rs:
                    blk_data.z[(k, rr, n)].set_value(1 if rr == 0 else 0)
                kn_seen.add((k, n))

    return m

In [44]:
def add_zero_sf_block(m, name, probability):
    block = pyo.Block()
    setattr(m, name, block)
    block = getattr(m, name)

    block.probability = float(probability)

    # SF for this block is exactly zero
    block.sf_expr = pyo.Expression(expr=0.0)

    # accumulate into global ESF
    if m.ESF.expr is None or m.ESF.is_fixed():
        m.ESF.set_value(block.probability * block.sf_expr)
    else:
        m.ESF.set_value(m.ESF.expr + block.probability * block.sf_expr)

    return block

In [45]:
def first_full_sf_block_name(m, prefix="sf", max_search=10000):
    for i in range(max_search):
        name = f"{prefix}{i}"
        blk = m.find_component(name)
        if blk is None:
            continue
        if blk.find_component("N") is not None and blk.find_component("theta") is not None:
            return name
    return None

In [46]:
nodes_u, weights_u = smolyak_nodes_weights(n_theta=len(t_bounds), level=4, rule="gaussian", growth=True)

m_sf_sm_exact = build_base_model_esf(d_bounds=d_bounds)

def pdf_expr(block, n):
    eps = 1e-6
    S = block.theta[0, n]
    D = block.theta[1, n]
    x = S - 8.0 + eps
    return (1.0 / (1.2 * math.pi * x)) * pyo.exp(
        -1.39 * (pyo.log(x))**2 - 0.5 * (D - 7.0)**2
    )

In [47]:
y_state_list

[(0, 0, 0),
 (0, 0, 1),
 (0, 1, 0),
 (1, 0, 0),
 (0, 1, 1),
 (1, 0, 1),
 (1, 1, 0),
 (1, 1, 1)]

In [48]:
for idx, state in enumerate(y_state_list):
    data = _prepare_state_data(state, solve_algo=mpqp_algorithm.geometric_parallel, theta_algo=mpqp_algorithm.combinatorial, log=True)

    if data is None:
        print(f"No valid theta regions for state {state}; skipping SF block")
        _ = add_zero_sf_block(m=m_sf_sm_exact, name=f"sf{idx}", probability=y_dict[state])
    else:
        _ = add_sf_block(m=m_sf_sm_exact, name=f"sf{idx}", theta_bounds_list=data["filtered_theta_bounds"], theta_regions_list=data["filtered_theta_regions"],
                         nodes_u=nodes_u, weights_u=weights_u, t_bounds=t_bounds, pdf_expr=pdf_expr, probability=y_dict[state], big_m=1e5)

    print(f"Finished for state {state}.")

Using a found active set [0, 1, 2, 3, 4, 6, 7, 8]
Spawned threads across 24
 Number of Facets to look at this time 10
Number of critical regions in for flexibility function for state (0, 0, 0): 1
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
No valid theta regions for state (0, 0, 0); skipping SF block
Finished for state (0, 0, 0).
Using a found active set [0, 1, 2, 3, 4, 6, 7, 9]
Spawned threads across 24
 Number of Facets to look at this time 12
 Number of Facets to look at this time 19
Number of critical regions in for flexibility function for state (0, 0, 1): 3
Finished solving for theta1
Finished solving for theta2
Finished for state (0, 0, 1).
Using a found active set [0, 1, 2, 3, 4, 6, 7, 8]
Spawned threads across 2

In [49]:
# for idx, state in enumerate(y_state_list):
#     state_model = create_flexibility_model(y_list=state)
#     state_prob = state_model.formulate_problem()
#     state_prob.process_constraints()
#     
#     state_flex_sol = solve_mpqp(problem=state_prob, algorithm=mpqp_algorithm.geometric_parallel)
#     print(f"Number of critical regions in for flexibility function for state {state}: {len(state_flex_sol.critical_regions)}")
#     start_time = time.time()
#     _, state_sol_list = get_theta_bounds(flex_sol=state_flex_sol, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds, mp_algo=mpqp_algorithm.combinatorial)
#     end_time = time.time()
#     print(f"Elapsed time for solving mp problems for state {state} : {end_time - start_time}")
#     
#     state_theta_bounds_list, state_theta_regions_list = get_bounds_regions(sols=state_sol_list)
#     
#     filtered_theta_bounds = []
#     filtered_theta_regions = []
#     for tb, tr in zip(state_theta_bounds_list, state_theta_regions_list):
#         if tb is None or tr is None:
#             continue
#         filtered_theta_bounds.append(tb)
#         filtered_theta_regions.append(tr)
#     
#     if not filtered_theta_bounds:
#         print(f"No valid theta regions for state {state}; skipping SF block")
#         _ = add_zero_sf_block(m=m_sf_sm_exact, name=f"sf{idx}", probability=y_dict[state])
#     else:
#         _ = add_sf_block(m=m_sf_sm_exact, name=f"sf{idx}", theta_bounds_list=state_theta_bounds_list, theta_regions_list=state_theta_regions_list, nodes_u=nodes_u, weights_u=weights_u, t_bounds=t_bounds, pdf_expr=pdf_expr, probability=y_dict[state], big_m=1e5)
# 
#     print(f"Finished for state {state}.")

In [50]:
set_esf_target(m_sf_sm_exact, target=0.2)
m_sf_sm_exact.qwerty = pyo.Var(bounds = (0.0, 10 * d_bounds[0][-1] + 3 * d_bounds[1][-1] + 10 * d_bounds[2][-1]))
m_sf_sm_exact.qwerty_con = pyo.Constraint(expr = m_sf_sm_exact.qwerty == 10 * m_sf_sm_exact.d0 + 3 * m_sf_sm_exact.d1 + 10 * m_sf_sm_exact.d2)
m_sf_sm_exact.obj = pyo.Objective(expr=m_sf_sm_exact.qwerty, sense=pyo.minimize)
# m_sf_sm_exact.obj = pyo.Objective(expr=m_sf_sm_exact.d0 + m_sf_sm_exact.d1 + m_sf_sm_exact.d2, sense=pyo.minimize)
# obj_ub = d_bounds[0][-1] + d_bounds[1][-1] + d_bounds[2][-1]  # compute from your d-bounds
# obj_ub = sum(i[-1] for i in d_bounds)  # sum of upper bounds
# m_sf_sm_exact.obj_ub = pyo.Constraint(
#     expr = m_sf_sm_exact.obj <= obj_ub
# )

In [51]:
# Warmstart entire Smolyak model
warmstart_exact_smolyak_model(
    m_sf_sm_exact,
    theta0_init=9.0,
    theta1_init=7.0,
    tmin_init=8.5,
    tmax_init=9.5,
    d_init=None,          # or a list of starting d_j values
    jac_floor=1e-6,
    prefer_existing=True, # keep any values already set on Vars
)

In [52]:
# blk_name = first_full_sf_block_name(m_sf_sm_exact)
# if blk_name is not None:
#     warmstart_exact_smolyak_minlp(m_sf_sm_exact, blk_name, d_init=[4,3,4])

In [53]:
def model_stats(m):
    from pyomo.core import Var, Constraint
    vars_ = list(m.component_data_objects(Var, descend_into=True))
    cons_ = list(m.component_data_objects(Constraint, active=True, descend_into=True))

    n_vars = len(vars_)
    n_bin = sum(v.is_binary() for v in vars_)
    n_int = sum(v.is_integer() for v in vars_ if not v.is_binary())
    n_cont = n_vars - n_bin - n_int

    return {"vars_total": n_vars, "vars_binary": n_bin, "vars_integer": n_int, "vars_continuous": n_cont, "constraints": len(cons_)}

In [54]:
print(model_stats(m_sf_sm_exact))

{'vars_total': 4294, 'vars_binary': 2750, 'vars_integer': 0, 'vars_continuous': 1544, 'constraints': 35587}


In [55]:
smolyak_solve_start = time.perf_counter()
res = pyo.SolverFactory("gams", solver="baron").solve(m_sf_sm_exact, tee=True, keepfiles=True, tmpdir=r"C:\gams_tmp", add_options = ['GAMS_MODEL.optfile = 1;','$onecho > baron.opt', 'EpsR 0.02', '$offecho'])
smolyak_solve_end = time.perf_counter()

--- Job model.gms Start 04/13/26 14:32:45 45.7.0 64fbf3ce WEX-WEI x86 64bit/MS Windows
--- Applying:
    C:\GAMS\45\gmsprmNT.txt
--- GAMS Parameters defined
    Input C:\gams_tmp\model.gms
    Output C:\gams_tmp\output.lst
    ScrDir C:\gams_tmp\225z\
    SysDir C:\GAMS\45\
    CurDir C:\gams_tmp\
    LogOption 3
Licensee: MUD - 30 User License                          G230830|0002AO-GEN
          Texas A&M University, Chemical Engineering                DC11194
          C:\GAMS\45\gamslice.txt
          License Admin: Jeff Polasek, j-polasek@tamu.edu                  
          The maintenance period of the license expired on Jun 25, 2024
          Please contact GAMS or your distributor for further information
Processor information: 1 socket(s), 16 core(s), and 24 thread(s) available
GAMS 45.7.0   Copyright (C) 1987-2024 GAMS Development. All rights reserved
--- Starting compilation
--- model.gms(82433) 17 Mb
--- $echo File C:\gams_tmp\baron.opt
--- model.gms(122375) 25 Mb
--- Start

In [56]:
m_sf_sm_exact.esf_target.pprint()

esf_target : Size=1, Index=None, Domain=Any, Default=None, Mutable=True
    Key  : Value
    None :   0.2


In [57]:
value(m_sf_sm_exact.ESF)

0.20000000000000073

In [58]:
value(m_sf_sm_exact.obj)

88.95140664951388

In [59]:
m_sf_sm_exact.d0.pprint()

d0 : Size=1, Index=None
    Key  : Lower : Value             : Upper : Fixed : Stale : Domain
    None :     0 : 3.836317135549876 :    10 : False : False :  Reals


In [60]:
m_sf_sm_exact.d1.pprint()

d1 : Size=1, Index=None
    Key  : Lower : Value             : Upper : Fixed : Stale : Domain
    None :     0 : 3.529411764671637 :    10 : False : False :  Reals


In [61]:
m_sf_sm_exact.d2.pprint()

d2 : Size=1, Index=None
    Key  : Lower : Value             : Upper : Fixed : Stale : Domain
    None :     0 : 4.000000000000021 :    10 : False : False :  Reals


In [62]:
design_vector = np.array([value(m_sf_sm_exact.d0), value(m_sf_sm_exact.d1), value(m_sf_sm_exact.d2)])

In [63]:
gl_esf, sm_esf = calculate_esf(y_d=y_dict, d_v=design_vector, n_q = 10, s_level = 8)

Spawned threads across 24
Time at depth test 1, 0.00032830238342285156
Number of active sets to be considered is 4
Time to run all tasks in parallel 0.005029916763305664
Time to process all depth outputs 0.0
Time at depth test 2, 0.005358219146728516
Number of active sets to be considered is 3
Time to run all tasks in parallel 0.0050351619720458984
Time to process all depth outputs 0.0
Time at depth test 3, 0.010393381118774414
Number of active sets to be considered is 2
Time to run all tasks in parallel 0.009001493453979492
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
No valid theta regions for state (0, 0, 0); skipping state
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 5


c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


Smolyak stochastic flexibility computed in 0.0080 seconds.
Finished for state (0, 0, 1).
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 4
Time to run all tasks in parallel 0.005996227264404297
Time to process all depth outputs 0.0
Time at depth test 2, 0.005996227264404297
Number of active sets to be considered is 3
Time to run all tasks in parallel 0.005177736282348633
Time to process all depth outputs 0.0
Time at depth test 3, 0.01117396354675293
Number of active sets to be considered is 2
Time to run all tasks in parallel 0.008517980575561523
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
No valid theta regions for state (0, 1, 0); skipping state
Spawned threads across 24
Ti

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


Smolyak stochastic flexibility computed in 0.0088 seconds.
Finished for state (0, 1, 1).
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 5
Time to run all tasks in parallel 0.006683826446533203
Time to process all depth outputs 0.0
Time at depth test 2, 0.006683826446533203
Number of active sets to be considered is 6
Time to run all tasks in parallel 0.008029937744140625
Time to process all depth outputs 0.0
Time at depth test 3, 0.014713764190673828
Number of active sets to be considered is 10
Time to run all tasks in parallel 0.017449140548706055
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 15
Time to run all tasks in parallel 0.016064167022705078
Time to process all depth outputs 0.0
Time at depth test 2, 0.016064167022705078
Number of active sets to be considered is 78
Time to run all tasks in parallel 0.07818055152893066
Time to process all depth outputs 0.0
Time at depth test 3, 0.09424

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


Spawned threads across 24
Time at depth test 1, 0.0009992122650146484
Number of active sets to be considered is 5
Time to run all tasks in parallel 0.005999565124511719
Time to process all depth outputs 0.0
Time at depth test 2, 0.006998777389526367
Number of active sets to be considered is 6
Time to run all tasks in parallel 0.00856924057006836
Time to process all depth outputs 0.0
Time at depth test 3, 0.015568017959594727
Number of active sets to be considered is 10
Time to run all tasks in parallel 0.017078161239624023
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 15
Time to run all tasks in parallel 0.016997814178466797
Time to process all depth outputs 0.0
Time at depth test 2, 0.016997814178466797
Number of active sets to be considered is 78
Time to run all tasks in parallel 0.07771706581115723
Time to process all depth outputs 0.0
Time at depth test 3, 0.09471487998962402
Number of active sets to be considered is 318
Time to run a

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 5
Time to run all tasks in parallel 0.006008148193359375
Time to process all depth outputs 0.0
Time at depth test 2, 0.006008148193359375
Number of active sets to be considered is 6
Time to run all tasks in parallel 0.007998228073120117
Time to process all depth outputs 0.0
Time at depth test 3, 0.014006376266479492
Number of active sets to be considered is 10
Time to run all tasks in parallel 0.0166013240814209
Spawned threads across 24
Time at depth test 1, 0.0
Number of active sets to be considered is 17
Time to run all tasks in parallel 0.01856541633605957
Time to process all depth outputs 0.0
Time at depth test 2, 0.01856541633605957
Number of active sets to be considered is 105
Time to run all tasks in parallel 0.053636789321899414
Time to process all depth outputs 0.0
Time at depth test 3, 0.07220220565795898
Number of active sets to be considered is 478
Time to run all tasks in paralle

c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:90: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  tmp = numpy_func(*[poly.values[keys[0]] for poly in inputs], **kwargs)
c:\Users\swaminathan.sundar\stochastic_flexibility\energiapy\.venv\Lib\site-packages\numpoly\dispatch.py:104: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  out_.values[key] = numpy_func(*[poly.values[key] for poly in inputs], **kwargs)


In [64]:
print(f'Gauss Legendre ESF: {gl_esf}')
print(f'Smolyak ESF: {sm_esf}')

Gauss Legendre ESF: 0.12237052154275893
Smolyak ESF: 0.12140091065373743


In [ ]:
def build_model_gl(sf_target: float):
    # --- you already have these pieces ---
    # sf_exprs_gl, sf_regions_gl, ... loaded from cache
    # design_bounds dict exists
    # preprocess_sf_expressions + embed_sf_constraints_to_model

    m = pyo.ConcreteModel()

    # (rebuild from cached expressions)
    sf_funcs_gl, region_funcs_gl, var_names_gl = preprocess_sf_expressions(
        expr_list=sf_exprs_gl,
        region_list=sf_regions_gl,
    )
    embed_sf_constraints_to_model(
        instance=m,
        sf_pyomo_funcs=sf_funcs_gl,
        region_pyomo_funcs=region_funcs_gl,
        var_names=var_names_gl,
        bounds_dict=design_bounds,
        big_m=1e4,
        initial_target=sf_target,
    )

    # objective (your example)
    # m.obj = pyo.Objective(expr=10*m.d0 - 10*m.d1, sense=pyo.minimize)
    m.obj = pyo.Objective(expr=10*m.d0 + 3*m.d1 + 10*m.d2, sense=pyo.minimize)
    # set target
    m.sf_target.set_value(sf_target)
    return m


def build_model_sm(sf_target: float):
    # your existing Smolyak MINLP builder
    m = build_exact_smolyak_sf_minlp(
        theta_bounds_list=theta_bounds_list,
        theta_regions_list=theta_regions_list,
        nodes_u=nodes_u,
        weights_u=weights_u,
        d_bounds=d_bounds,
        t_bounds=t_bounds,
        sf_target=sf_target,
        big_m=1e4,
        pdf_expr=pdf_expr,
        # If you changed to pdf_expr input:
        # pdf_expr=pdf_algae or pdf_bansal
    )

    # m.obj = pyo.Objective(expr=10*m.d0 - 10*m.d1, sense=pyo.minimize)
    m.obj = pyo.Objective(expr=10*m.d0 + 3*m.d1 + 10*m.d2, sense=pyo.minimize)
    # warmstart (important for log-type PDFs)
    warmstart_exact_smolyak_minlp(m)
    return m

In [ ]:
def _safe_value(x):
    try:
        return float(pyo.value(x))
    except Exception:
        return None


def _get_design_vars(m, prefix="d"):
    out = {}
    j = 0
    while hasattr(m, f"{prefix}{j}"):
        out[f"{prefix}{j}"] = _safe_value(getattr(m, f"{prefix}{j}"))
        j += 1
    return out


def solve_one(m, solver="gams", solver_name="baron", tee=False, tmpdir=r"C:\gams_tmp"):
    t0 = time.perf_counter()
    res = pyo.SolverFactory(solver, solver=solver_name).solve(
        m, tee=tee, keepfiles=False, tmpdir=tmpdir
    )
    t1 = time.perf_counter()

    # Try to pull common fields robustly
    obj = _safe_value(m.obj)
    sf_target = _safe_value(m.sf_target) if hasattr(m, "sf_target") else None

    # achieved SF could be Expression m.SF or Var m.sf
    sf_ach = None
    if hasattr(m, "SF"):
        sf_ach = _safe_value(m.SF)
    elif hasattr(m, "sf"):
        sf_ach = _safe_value(m.sf)

    row = {
        "solve_time_s": t1 - t0,
        "obj": obj,
        "sf_target": sf_target,
        "sf_achieved": sf_ach,
        "termination": str(getattr(res.solver, "termination_condition", "")),
        "status": str(getattr(res.solver, "status", "")),
    }
    row.update(_get_design_vars(m, "d"))
    return row


def sweep_targets(
    targets,
    *,
    build_gl,
    build_sm,
    tee=False
):
    records = []

    for t in tqdm(targets, desc="SF target sweep", unit="target"):
        # ---------- Gaussian–Legendre ----------
        try:
            m_gl = build_gl(t)
            r_gl = solve_one(m_gl, tee=tee)
            r_gl["method"] = "GL"
        except Exception as e:
            r_gl = {"method": "GL", "sf_target": t, "error": repr(e)}
        records.append(r_gl)

        # ---------- Smolyak ----------
        try:
            m_sm = build_sm(t)
            r_sm = solve_one(m_sm, tee=tee)
            r_sm["method"] = "Smolyak"
        except Exception as e:
            r_sm = {"method": "Smolyak", "sf_target": t, "error": repr(e)}
        records.append(r_sm)

    df = pd.DataFrame(records)
    for c in ["solve_time_s", "obj", "sf_target", "sf_achieved"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

In [ ]:
# targets = [0.50, 0.60, 0.70, 0.80, 0.90, 0.98]
targets = [0.1, 0.2, 0.3, 0.4, 0.5]
df = sweep_targets(
    targets,
    build_gl=build_model_gl,
    build_sm=build_model_sm,
    tee=True
)

df.to_csv("sf_pareto_gl_vs_sm.csv", index=False)
df

In [ ]:
PLOT_DIR = "figures"
os.makedirs(PLOT_DIR, exist_ok=True)

sns.set_theme(
    style="whitegrid",
    context="paper",
    font_scale=1.3
)


def _save(fig, name):
    fig.tight_layout()
    fig.savefig(f"{PLOT_DIR}/{name}.pdf", bbox_inches="tight")
    fig.savefig(f"{PLOT_DIR}/{name}.png", dpi=300, bbox_inches="tight")


def plot_pareto(df):
    fig, ax = plt.subplots(figsize=(6, 4))

    for method, marker in [("GL", "o"), ("Smolyak", "s")]:
        sub = df[df["method"] == method]
        ax.plot(
            sub["sf_target"],
            sub["obj"],
            marker=marker,
            linewidth=2,
            label=method
        )

    ax.set_xlabel("SF target")
    ax.set_ylabel("Objective value")
    ax.legend()
    ax.set_title("Pareto Front: Objective vs SF")

    _save(fig, "pareto_sf_vs_objective")
    return fig, ax


def plot_design_vs_target(df, var):
    fig, ax = plt.subplots(figsize=(6, 4))

    for method, marker in [("GL", "o"), ("Smolyak", "s")]:
        sub = df[df["method"] == method]
        ax.plot(
            sub["sf_target"],
            sub[var],
            marker=marker,
            linewidth=2,
            label=method
        )

    ax.set_xlabel("SF target")
    ax.set_ylabel(var)
    ax.legend()
    ax.set_title(f"{var} vs SF target")

    _save(fig, f"{var}_vs_sf_target")
    return fig, ax


def plot_accuracy(df):
    fig, ax = plt.subplots(figsize=(6, 4))

    for method, marker in [("GL", "o"), ("Smolyak", "s")]:
        sub = df[df["method"] == method]
        err = sub["sf_achieved"] - sub["sf_target"]
        ax.plot(
            sub["sf_target"],
            err,
            marker=marker,
            linewidth=2,
            label=method
        )

    ax.axhline(0, color="k", linestyle="--", linewidth=1)
    ax.set_xlabel("SF target")
    ax.set_ylabel("SF achieved − SF target")
    ax.legend()
    ax.set_title("SF Accuracy")

    _save(fig, "sf_accuracy")
    return fig, ax


def plot_runtime(df):
    fig, ax = plt.subplots(figsize=(6, 4))

    for method, marker in [("GL", "o"), ("Smolyak", "s")]:
        sub = df[df["method"] == method]
        ax.plot(
            sub["sf_target"],
            sub["solve_time_s"],
            marker=marker,
            linewidth=2,
            label=method
        )

    ax.set_yscale("log")
    ax.set_xlabel("SF target")
    ax.set_ylabel("Solve time [s]")
    ax.legend()
    ax.set_title("Solve Time vs SF Target")

    _save(fig, "runtime_vs_sf_target")
    return fig, ax


def plot_runtime_summary(df):
    fig, ax = plt.subplots(figsize=(5, 4))

    summary = (
        df.groupby("method")["solve_time_s"]
        .mean()
        .reset_index()
    )

    sns.barplot(
        data=summary,
        x="method",
        y="solve_time_s",
        ax=ax
    )

    ax.set_yscale("log")
    ax.set_ylabel("Mean solve time [s]")
    ax.set_xlabel("")
    ax.set_title("Average Computational Cost")

    _save(fig, "runtime_summary")
    return fig, ax

In [ ]:
plot_pareto(df)

for col in sorted(c for c in df.columns if c.startswith("d") and c[1:].isdigit()):
    plot_design_vs_target(df, var=col)

plot_accuracy(df)
plot_runtime(df)
plot_runtime_summary(df)

In [ ]:
def summary_table(df):
    ok = _prep(df)
    ok["sf_gap_abs"] = (ok["sf_achieved"] - ok["sf_target"]).abs()
    agg = ok.groupby("method").agg(
        n=("obj", "count"),
        time_mean=("solve_time_s", "mean"),
        time_median=("solve_time_s", "median"),
        time_max=("solve_time_s", "max"),
        sf_gap_mean=("sf_gap_abs", "mean"),
        sf_gap_max=("sf_gap_abs", "max"),
    ).reset_index()
    return agg


summary = summary_table(df)
summary.to_csv("sf_gl_vs_sm_summary.csv", index=False)
summary